# Job Preprocessing Pipeline v3 — PhoBERT + Cosine Similarity

Mục tiêu:
- Chuẩn hoá dữ liệu job posting
- Trích xuất metadata + skills
- Tạo text tối ưu cho PhoBERT
- Sinh embedding cho matching / retrieval
- Dùng cosine similarity cho semantic search
- Xuất artifact cho matching, chatbot, section retrieval

In [73]:
import re
import os
import json
import math
import html
import time
import unicodedata
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

# Optional: để word segmentation tiếng Việt
try:
    from underthesea import word_tokenize
    HAS_UNDERTHESEA = True
except Exception:
    HAS_UNDERTHESEA = False

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)

NOTEBOOK_VERSION = "preprocessing_v3_phobert"
# RAW_INPUT_PATH = Path("../data/raw/jobs/topcv_all_fields_merged_2026-03-21_19-27-47")   # ← sửa
RAW_INPUT_PATH = Path("../../data/raw/jobs/topcv_all_fields_merged_2026-03-21_19-27-47.csv")
ARTIFACT_DIR = Path("../outputs_preprocessing_v3")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

RUN_EMBEDDING = True
RUN_SECTION_EMBEDDING = True
SAVE_INTERMEDIATE = True

PHOBERT_MODEL_NAME = "vinai/phobert-base"
PHOBERT_MAX_LENGTH_MATCH = 256
PHOBERT_MAX_LENGTH_CHATBOT = 320
PHOBERT_MAX_LENGTH_CHUNK = 256
PHOBERT_BATCH_SIZE = 16
NORMALIZE_EMBEDDINGS = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("NOTEBOOK_VERSION:", NOTEBOOK_VERSION)
print("RAW_INPUT_PATH:", RAW_INPUT_PATH)
print("ARTIFACT_DIR:", ARTIFACT_DIR.resolve())
print("RUN_EMBEDDING:", RUN_EMBEDDING)
print("RUN_SECTION_EMBEDDING:", RUN_SECTION_EMBEDDING)
print("PHOBERT_MODEL_NAME:", PHOBERT_MODEL_NAME)
print("DEVICE:", DEVICE)
print("HAS_UNDERTHESEA:", HAS_UNDERTHESEA)

NOTEBOOK_VERSION: preprocessing_v3_phobert
RAW_INPUT_PATH: ..\..\data\raw\jobs\topcv_all_fields_merged_2026-03-21_19-27-47.csv
ARTIFACT_DIR: D:\TTTN\project_v2\notebooks\outputs_preprocessing_v3
RUN_EMBEDDING: True
RUN_SECTION_EMBEDDING: True
PHOBERT_MODEL_NAME: vinai/phobert-base
DEVICE: cpu
HAS_UNDERTHESEA: True


In [74]:
# Chuẩn hóa giá trị rỗng
def normalize_empty_value(val):
    if val is None:
        return None
    if isinstance(val, float) and pd.isna(val):
        return None

    s = str(val).strip()
    if not s:
        return None

    lowered = s.lower().strip()
    empty_tokens = {
        "nan", "none", "null", "n/a", "na", "-", "--", "---",
        "không", "không rõ", "chưa rõ", "chưa cập nhật", "đang cập nhật",
        "not specified", "unknown"
    }
    return None if lowered in empty_tokens else s

# Chuẩn hóa thành chuỗi
def safe_str(x):
    x = normalize_empty_value(x)
    return "" if x is None else str(x)

# Lấy Series từ DataFrame, nếu không có thì trả về Series mặc định
def get_series(df, col, default=None):
    if col in df.columns:
        return df[col]
    if default is None:
        default = [None] * len(df)
    return pd.Series(default, index=df.index)

# Lấy giá trị đầu tiên không rỗng từ danh sách các giá trị
def first_non_empty(*values):
    for v in values:
        v = normalize_empty_value(v)
        if v is not None:
            return v
    return None

# Chuẩn hóa unicode
def normalize_unicode(text):
    text = safe_str(text)
    return unicodedata.normalize("NFC", text)

# Xóa thẻ html
def remove_html(text):
    text = safe_str(text)
    text = html.unescape(text)
    text = re.sub(r"<br\s*/?>", "\n", text, flags=re.I)
    text = re.sub(r"</p\s*>", "\n", text, flags=re.I)
    text = re.sub(r"<[^>]+>", " ", text)
    return text

# Chuẩn hóa các kiểu gạch đầu dòng khác nhau về "-"
def normalize_dash(text):
    text = safe_str(text)
    dash_map = {
        "–": "-",
        "—": "-",
        "−": "-",
        "•": "-",
        "●": "-",
        "▪": "-",
        "►": "-",
        "✅": "-",
        "✔": "-",
    }
    for k, v in dash_map.items():
        text = text.replace(k, v)
    return text

# Loại bỏ phần tử trùng lặp trong list, giữ nguyên thứ tự
def deduplicate_list(values):
    out = []
    seen = set()
    for v in values:
        key = safe_str(v).strip()
        if not key:
            continue
        if key not in seen:
            seen.add(key)
            out.append(v)
    return out

# Loại bỏ dòng trùng lặp trong văn bản, giữ nguyên thứ tự và cấu trúc cơ bản
def deduplicate_text_lines(text, min_key_len=12):
    text = safe_str(text)
    if not text:
        return ""
    out_lines = []
    seen = set()
    for line in text.splitlines():
        raw = line.strip()
        if not raw:
            continue
        key = re.sub(r"\s+", " ", raw.lower())
        if len(key) < min_key_len:
            out_lines.append(raw)
            continue
        if key not in seen:
            seen.add(key)
            out_lines.append(raw)
    return "\n".join(out_lines).strip()

# Các hàm làm sạch văn bản ở nhiều mức độ khác nhau
# Làm sạch nhẹ, giữ nguyên cấu trúc cơ bản (giữ format gốc, chỉ loại bỏ những thứ thực sự không cần thiết)
def clean_text_light(text):
    text = normalize_empty_value(text)
    if text is None:
        return ""

    text = normalize_unicode(text)
    text = remove_html(text)
    text = normalize_dash(text)
    text = text.replace("\\n", "\n")
    text = re.sub(r"[\u200b-\u200f\uFEFF]", "", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r" ?\n ?", "\n", text)
    text = deduplicate_text_lines(text)
    return text.strip()

# Làm sạch nghiêm ngặt, loại bỏ hầu hết dấu câu, chỉ giữ lại chữ và số (giữ nguyên cấu trúc cơ bản như xuống dòng và gạch đầu dòng để preserve meaning)
def clean_text_preserve_structure(text):
    text = clean_text_light(text)
    if not text:
        return ""
    text = re.sub(r"[ ]{2,}", " ", text)
    text = re.sub(r"\n\s*-\s*", "\n- ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

# Làm sạch nghiêm ngặt hơn, chỉ giữ lại chữ, số và một số dấu câu cơ bản (để vector hóa và matching)
def clean_text_strict(text):
    text = clean_text_light(text)
    if not text:
        return ""
    text = text.lower() # chuyển về chữ thường để chuẩn hóa
    text = re.sub(r"[^\w\sàáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ/+\-#\.]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# Làm sạch cho PhoBERT, giữ lại dấu câu cần thiết để preserve meaning 
def clean_text_for_phobert(text):
    text = normalize_empty_value(text)
    if text is None:
        return ""

    text = normalize_unicode(text)
    text = remove_html(text)
    text = normalize_dash(text)
    text = text.replace("\\n", "\n")

    text = re.sub(r"[\u200b-\u200f\uFEFF]", "", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    # giữ dấu câu đủ để preserve meaning cho PhoBERT
    text = re.sub(r"[^\w\sÀ-ỹ\.,;:/\-\+\#\(\)%\n]", " ", text)
    text = re.sub(r" +", " ", text)
    text = re.sub(r" ?\n ?", "\n", text)

    return text.strip()

# Rút gọn văn bản theo số lượng từ, giữ nguyên cấu trúc cơ bản
def truncate_by_words(text, max_words=220):
    text = safe_str(text)
    if not text:
        return ""
    words = text.split()
    return " ".join(words[:max_words]).strip()

# Lưu DataFrame thành Parquet nếu có thể, nếu không thì lưu CSV
def save_table(df, base_path: Path):
    base_path = Path(base_path)
    try:
        out_path = str(base_path) + ".parquet"
        df.to_parquet(out_path, index=False)
        return out_path
    except Exception:
        out_path = str(base_path) + ".csv"
        df.to_csv(out_path, index=False, encoding="utf-8-sig")
        return out_path

In [75]:
def load_raw_data(input_path: Path) -> pd.DataFrame:
    if not input_path.exists():
        raise FileNotFoundError(f"Không tìm thấy file: {input_path}")

    suffix = input_path.suffix.lower()
    if suffix in [".xlsx", ".xls"]:
        df = pd.read_excel(input_path)
    elif suffix == ".csv":
        df = pd.read_csv(input_path)
    else:
        raise ValueError(f"Unsupported file format: {suffix}")

    print(f"[INFO] Loaded raw data: {df.shape[0]} rows x {df.shape[1]} cols")
    return df


df_raw = load_raw_data(RAW_INPUT_PATH)
display(df_raw.head(3))
print(df_raw.columns.tolist())

[INFO] Loaded raw data: 369 rows x 33 cols


,job_url,source_field_name,field_count,title,detail_title,company_name,company_name_full,company_url,company_url_from_job,salary_list,detail_salary,address_list,detail_location,exp_list,detail_experience,deadline,tags,job_level,education_level,job_quantity,employment_type,working_addresses,working_times,desc_mota,desc_yeucau,desc_quyenloi,company_website,company_scale_from_job,company_scale,company_field_from_job,company_address_from_job,company_address,company_description
0,https://www.topcv.vn/brand/beeogisticscorporation/tuyen-dung/junior-fpa-analyst-ho-chi-minh-j1851113.html,Data Analyst,1,Junior FP&A Analyst - Hồ Chí Minh,Junior FP&A Analyst - Hồ Chí Minh,Bee Logistics Corporation,Bee Logistics Corporation,https://www.topcv.vn/brand/beeogisticscorporation?id=2450,https://www.topcv.vn/brand/beeogisticscorporation?id=2450,12 - 20 triệu,12 - 20 triệu,Hồ Chí Minh (mới),Hồ Chí Minh,2 năm,2 năm,27/03/2026,Chuyên môn Data Analyst; Tài chính; Kế toán,Nhân viên,Đại Học trở lên,1 người,Toàn thời gian,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hồ Chí Minh: TTC 253 Hoàng Văn Thụ, Phường Tân Sơn Hòa (quận Tân Bình cũ)",Thứ 2 - Thứ 6 (từ 08:00 đến 17:30) Thứ 7 (từ 08:00 đến 12:00) Nghỉ 2 buổi sáng Thứ 7/tháng\nThứ 2 - Thứ 6 (từ 08:00 đến 17:30)\nThứ 7 (từ 08:00 đến 12:00)\nNghỉ 2 buổi sáng Thứ 7/tháng,"– Overseeing all financial planning, reporting & analysis for the Bee office. – Track management reports focusing on Budget/Estimate/Actuals to ensure those are delivered on time and with high-qua...",– At least 2 year’ experience in fthe inance/accounting area within big/complex organizations and/or audit services in reputable finance consulting firms. – Bachelor's degree in Finance/Accounting...,"– Competitive salary according to personal experience and ability – Lunch allowance, phone allowance – Salary increase and annual bonus on holidays (2/9, 1/5, Mid-Autumn Festival, 1/6, New Year, L...",http://www.beelogistics.com/,NaN,500-1000,NaN,NaN,"Addr: 11th Floor, TTC Tower, 253 Hoang Van Thu, Ward 2, Tan Binh District, Ho Chi Minh City, Viet Nam Addr: 10th Floor, 789 Tower, 147 Hoang Quoc Viet, Nghia Do Ward, Cau Giay District, Hanoi City...","Xuất phát với ước mơ xây dựng một doanh nghiệp Việt, có thể cung cấp, phát triển dịch vụ logistics toàn cầu, tin cậy dựa trên chất lượng dịch vụ, con người và công nghệ, cùng niềm đam mê với nghề ..."
1,https://www.topcv.vn/brand/educa/tuyen-dung/data-governance-specialist-j2055051.html,Data Analyst,1,Data Governance Specialist,Data Governance Specialist,EDUPIA,EDUPIA,https://www.topcv.vn/brand/educa?id=17270,https://www.topcv.vn/brand/educa?id=17270,20 - 30 triệu,20 - 30 triệu,Hà Nội,NaN,2 năm,2 năm,Còn 12 ngày để ứng tuyển,Chuyên môn Data Analyst,Nhân viên,Đại Học trở lên,1 người,Toàn thời gian,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Tầng 2-3-5-6 Tòa Comatce, 61 Ngụy Như Kon Tum, Phường Thanh Xuân (quận Thanh Xuân cũ)\n...",Thứ 2 - Thứ 6 (từ 08:00 đến 17:30) Thứ 7 (từ 08:00 đến 12:00)\nThứ 2 - Thứ 6 (từ 08:00 đến 17:30)\nThứ 7 (từ 08:00 đến 12:00),"− Xây dựng, triển khai và duy trì khung Data Governance cho toàn hệ thống dữ liệu của Công ty. − Xây dựng chính sách, tiêu chuẩn, quy định về quản lý dữ liệu (data quality, data profiling, data se...","− Đã tốt nghiệp ĐH trở lên, ưu tiên các chuyên ngành về Công nghệ thông tin/ Hệ thống thông tin quản lý hoặc các lĩnh vực liên quan và tối thiểu 2 năm kinh nghiệm trong lĩnh vực Data Governance/ D...","Mức lương thỏa thuận theo năng lực. Thử việc hưởng 100% lương trong 2 tháng. Thưởng Lễ/Tết, thưởng hiệu quả làm việc: theo quy định của Công ty. Chính sách đánh giá điều chỉnh lương hàng năm minh ...",https://edupia.vn/,NaN,500-1000,NaN,NaN,"Trụ sở: Tầng 2,3,5,6 - Tòa Vincem Comatce Tower, số 61 Ngụy Như Kon Tum, Thanh Xuân, Hà Nội. Chi nhánh: Tầng 6, Tòa nhà Báo Sinh Viên - Hoa Học Trò, Yên Hòa, Cầu Giấy, Hà Nội","Được thành lập

['job_url', 'source_field_name', 'field_count', 'title', 'detail_title', 'company_name', 'company_name_full', 'company_url', 'company_url_from_job', 'salary_list', 'detail_salary', 'address_list', 'detail_location', 'exp_list', 'detail_experience', 'deadline', 'tags', 'job_level', 'education_level', 'job_quantity', 'employment_type', 'working_addresses', 'working_times', 'desc_mota', 'desc_yeucau', 'desc_quyenloi', 'company_website', 'company_scale_from_job', 'company_scale', 'company_field_from_job', 'company_address_from_job', 'company_address', 'company_description']


In [76]:
# Import dữ liệu vào schema chung
def merge_semantic_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)

    out["job_url"] = get_series(df, "job_url")
    out["job_id"] = get_series(df, "job_id")

    out["job_title_raw"] = [
        first_non_empty(a, b)
        for a, b in zip(
            get_series(df, "detail_title"),
            get_series(df, "title")
        )
    ]
    out["job_slug_raw"] = get_series(df, "source_field_name")

    out["salary_raw"] = [
        first_non_empty(a, b)
        for a, b in zip(
            get_series(df, "detail_salary"),
            get_series(df, "salary_list")
        )
    ]

    out["location_raw"] = [
        first_non_empty(a, b)
        for a, b in zip(
            get_series(df, "address_list"),
            get_series(df, "detail_location")
        )
    ]

    out["working_addresses_raw"] = get_series(df, "working_addresses")

    out["working_times_raw"] = get_series(df, "working_times")

    out["experience_raw"] = [
        first_non_empty(a, b)
        for a, b in zip(
            get_series(df, "exp_list"),
            get_series(df, "detail_experience")
        )
    ]
    out["education_level_raw"] = get_series(df, "education_level")
    out["employment_type_raw"] = get_series(df, "employment_type")
    out["job_level_raw"] = get_series(df, "job_level")
    out["job_quantity_raw"] = get_series(df, "job_quantity")

    out["description_raw"] = get_series(df, "desc_mota")
    out["requirements_raw"] = get_series(df, "desc_yeucau")
    out["benefits_raw"] = get_series(df, "desc_quyenloi")

    out["tags_raw"] = get_series(df, "tags")

    out["deadline_raw"] = get_series(df, "deadline")

    out["company_name_raw"] = [
        first_non_empty(a, b)
        for a, b in zip(
            get_series(df, "company_name_full"),
            get_series(df, "company_name")
        )
    ]
    out["company_website_raw"] = get_series(df, "company_website")
    out["company_field_raw"] = get_series(df, "company_field_from_job")

    out["company_scale_raw"] = [
        first_non_empty(a, b)
        for a, b in zip(
            get_series(df, "company_scale_from_job"),
            get_series(df, "company_scale")
        )
    ]
    
    out["company_address_raw"] = [
        first_non_empty(a, b)
        for a, b in zip(
            get_series(df, "company_address_from_job"),
            get_series(df, "company_address")
        )
    ]
    out["company_description_raw"] = get_series(df, "company_description")

    return out


df = merge_semantic_columns(df_raw)
print("[INFO] After merging:", df.shape)
display(df.head(3))
df.info()

[INFO] After merging: (369, 24)


,job_url,job_id,job_title_raw,job_slug_raw,salary_raw,location_raw,working_addresses_raw,working_times_raw,experience_raw,education_level_raw,employment_type_raw,job_level_raw,job_quantity_raw,description_raw,requirements_raw,benefits_raw,tags_raw,deadline_raw,company_name_raw,company_website_raw,company_field_raw,company_scale_raw,company_address_raw,company_description_raw
0,https://www.topcv.vn/brand/beeogisticscorporation/tuyen-dung/junior-fpa-analyst-ho-chi-minh-j1851113.html,None,Junior FP&A Analyst - Hồ Chí Minh,Data Analyst,12 - 20 triệu,Hồ Chí Minh (mới),"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hồ Chí Minh: TTC 253 Hoàng Văn Thụ, Phường Tân Sơn Hòa (quận Tân Bình cũ)",Thứ 2 - Thứ 6 (từ 08:00 đến 17:30) Thứ 7 (từ 08:00 đến 12:00) Nghỉ 2 buổi sáng Thứ 7/tháng\nThứ 2 - Thứ 6 (từ 08:00 đến 17:30)\nThứ 7 (từ 08:00 đến 12:00)\nNghỉ 2 buổi sáng Thứ 7/tháng,2 năm,Đại Học trở lên,Toàn thời gian,Nhân viên,1 người,"– Overseeing all financial planning, reporting & analysis for the Bee office. – Track management reports focusing on Budget/Estimate/Actuals to ensure those are delivered on time and with high-qua...",– At least 2 year’ experience in fthe inance/accounting area within big/complex organizations and/or audit services in reputable finance consulting firms. – Bachelor's degree in Finance/Accounting...,"– Competitive salary according to personal experience and ability – Lunch allowance, phone allowance – Salary increase and annual bonus on holidays (2/9, 1/5, Mid-Autumn Festival, 1/6, New Year, L...",Chuyên môn Data Analyst; Tài chính; Kế toán,27/03/2026,Bee Logistics Corporation,http://www.beelogistics.com/,NaN,500-1000,"Addr: 11th Floor, TTC Tower, 253 Hoang Van Thu, Ward 2, Tan Binh District, Ho Chi Minh City, Viet Nam Addr: 10th Floor, 789 Tower, 147 Hoang Quoc Viet, Nghia Do Ward, Cau Giay District, Hanoi City...","Xuất phát với ước mơ xây dựng một doanh nghiệp Việt, có thể cung cấp, phát triển dịch vụ logistics toàn cầu, tin cậy dựa trên chất lượng dịch vụ, con người và công nghệ, cùng niềm đam mê với nghề ..."
1,https://www.topcv.vn/brand/educa/tuyen-dung/data-governance-specialist-j2055051.html,None,Data Governance Specialist,Data Analyst,20 - 30 triệu,Hà Nội,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Tầng 2-3-5-6 Tòa Comatce, 61 Ngụy Như Kon Tum, Phường Thanh Xuân (quận Thanh Xuân cũ)\n...",Thứ 2 - Thứ 6 (từ 08:00 đến 17:30) Thứ 7 (từ 08:00 đến 12:00)\nThứ 2 - Thứ 6 (từ 08:00 đến 17:30)\nThứ 7 (từ 08:00 đến 12:00),2 năm,Đại Học trở lên,Toàn thời gian,Nhân viên,1 người,"− Xây dựng, triển khai và duy trì khung Data Governance cho toàn hệ thống dữ liệu của Công ty. − Xây dựng chính sách, tiêu chuẩn, quy định về quản lý dữ liệu (data quality, data profiling, data se...","− Đã tốt nghiệp ĐH trở lên, ưu tiên các chuyên ngành về Công nghệ thông tin/ Hệ thống thông tin quản lý hoặc các lĩnh vực liên quan và tối thiểu 2 năm kinh nghiệm trong lĩnh vực Data Governance/ D...","Mức lương thỏa thuận theo năng lực. Thử việc hưởng 100% lương trong 2 tháng. Thưởng Lễ/Tết, thưởng hiệu quả làm việc: theo quy định của Công ty. Chính sách đánh giá điều chỉnh lương hàng năm minh ...",Chuyên môn Data Analyst,Còn 12 ngày để ứng tuyển,EDUPIA,https://edupia.vn/,NaN,500-1000,"Trụ sở: Tầng 2,3,5,6 - Tòa Vincem Comatce Tower, số 61 Ngụy Như Kon Tum, Thanh Xuân, Hà Nội. Chi nhánh: Tầng 6, Tòa nhà Báo Sinh Viên - Hoa Học Trò, Yên Hòa, Cầu Giấy, Hà Nội","Được thành lập năm 2018, Edupia là Edtech lớn nhất Việt Nam về Tiếng Anh Online cho trẻ em, đồng thời là Top 50 Edtech Nổi bật nhất Đông Nam Á năm 2022 & 2023. Qua nhiều năm phát triển, Edupia đến..."
2,https://www.topcv.vn/brand/educa/tuyen-dung/project-manager-du-an-ai-hub-j2067747.html,None,Project Manager Dự Án AI HUB,AI Engineer,30 - 35 triệu,Hà Nội,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Tầng 2,

<class 'pandas.DataFrame'>
RangeIndex: 369 entries, 0 to 368
Data columns (total 24 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   job_url                  369 non-null    str   
 1   job_id                   0 non-null      object
 2   job_title_raw            369 non-null    str   
 3   job_slug_raw             369 non-null    str   
 4   salary_raw               369 non-null    str   
 5   location_raw             369 non-null    str   
 6   working_addresses_raw    369 non-null    str   
 7   working_times_raw        339 non-null    str   
 8   experience_raw           369 non-null    str   
 9   education_level_raw      369 non-null    str   
 10  employment_type_raw      369 non-null    str   
 11  job_level_raw            369 non-null    str   
 12  job_quantity_raw         369 non-null    str   
 13  description_raw          369 non-null    str   
 14  requirements_raw         369 non-null    str   
 15  

In [77]:
# Kiểm tra tỷ lệ trùng lặp, tỷ lệ missing, và phân loại ngôn ngữ
def detect_language_type(text: str) -> str:
    text = safe_str(text)
    if not text:
        return "empty"

    has_vi = bool(re.search(r"[ăâêôơưđàáạảãèéẹẻẽìíịỉĩòóọỏõùúụủũỳýỵỷỹ]", text.lower()))
    has_en = bool(re.search(r"[a-z]", text.lower()))

    if has_vi and has_en:
        return "mixed"
    if has_vi:
        return "vi"
    if has_en:
        return "en"
    return "other"


audit_rows = []
combined_text = (
    get_series(df, "job_title_raw", "") .fillna("") + " " +
    get_series(df, "description_raw", "") .fillna("") + " " +
    get_series(df, "requirements_raw", "") .fillna("")
)

lang_type = combined_text.apply(detect_language_type)

audit_rows.append({
    "n_rows": len(df),
    "dup_by_url_ratio": df["job_url"].duplicated().mean() if "job_url" in df.columns else np.nan,
    "has_minimum_content_ratio": (
        (df["job_title_raw"].fillna("").str.len() > 0) &
        (
            (df["description_raw"].fillna("").str.len() > 0) |
            (df["requirements_raw"].fillna("").str.len() > 0)
        )
    ).mean(),
    "mixed_ratio": (lang_type == "mixed").mean(),
    "en_ratio": (lang_type == "en").mean(),
    "vi_ratio": (lang_type == "vi").mean(),
})

audit_df = pd.DataFrame(audit_rows)
missing_df = pd.DataFrame({
    "column": df.columns,
    "missing_ratio": [df[c].isna().mean() for c in df.columns]
}).sort_values("missing_ratio", ascending=False)

display(audit_df)
display(missing_df)

,n_rows,dup_by_url_ratio,has_minimum_content_ratio,mixed_ratio,en_ratio,vi_ratio
0,369,0.0,1.0,0.831978,0.168022,0.0


,column,missing_ratio
1,job_id,1.000000
19,company_website_raw,0.173442
7,working_times_raw,0.081301
20,company_field_raw,0.075881
23,company_description_raw,0.029810
3,job_slug_raw,0.000000
2,job_title_raw,0.000000
0,job_url,0.000000
6,working_addresses_raw,0.000000
5,location_raw,0.000000


In [78]:
df_clean = df.copy()

for col in df_clean.columns:
    df_clean[col] = df_clean[col].apply(normalize_empty_value)

print("df_raw shape :", df.shape)
print("df_clean shape:", df_clean.shape)
display(df_clean.head(3))

df_raw shape : (369, 24)
df_clean shape: (369, 24)


,job_url,job_id,job_title_raw,job_slug_raw,salary_raw,location_raw,working_addresses_raw,working_times_raw,experience_raw,education_level_raw,employment_type_raw,job_level_raw,job_quantity_raw,description_raw,requirements_raw,benefits_raw,tags_raw,deadline_raw,company_name_raw,company_website_raw,company_field_raw,company_scale_raw,company_address_raw,company_description_raw
0,https://www.topcv.vn/brand/beeogisticscorporation/tuyen-dung/junior-fpa-analyst-ho-chi-minh-j1851113.html,None,Junior FP&A Analyst - Hồ Chí Minh,Data Analyst,12 - 20 triệu,Hồ Chí Minh (mới),"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hồ Chí Minh: TTC 253 Hoàng Văn Thụ, Phường Tân Sơn Hòa (quận Tân Bình cũ)",Thứ 2 - Thứ 6 (từ 08:00 đến 17:30) Thứ 7 (từ 08:00 đến 12:00) Nghỉ 2 buổi sáng Thứ 7/tháng\nThứ 2 - Thứ 6 (từ 08:00 đến 17:30)\nThứ 7 (từ 08:00 đến 12:00)\nNghỉ 2 buổi sáng Thứ 7/tháng,2 năm,Đại Học trở lên,Toàn thời gian,Nhân viên,1 người,"– Overseeing all financial planning, reporting & analysis for the Bee office. – Track management reports focusing on Budget/Estimate/Actuals to ensure those are delivered on time and with high-qua...",– At least 2 year’ experience in fthe inance/accounting area within big/complex organizations and/or audit services in reputable finance consulting firms. – Bachelor's degree in Finance/Accounting...,"– Competitive salary according to personal experience and ability – Lunch allowance, phone allowance – Salary increase and annual bonus on holidays (2/9, 1/5, Mid-Autumn Festival, 1/6, New Year, L...",Chuyên môn Data Analyst; Tài chính; Kế toán,27/03/2026,Bee Logistics Corporation,http://www.beelogistics.com/,NaN,500-1000,"Addr: 11th Floor, TTC Tower, 253 Hoang Van Thu, Ward 2, Tan Binh District, Ho Chi Minh City, Viet Nam Addr: 10th Floor, 789 Tower, 147 Hoang Quoc Viet, Nghia Do Ward, Cau Giay District, Hanoi City...","Xuất phát với ước mơ xây dựng một doanh nghiệp Việt, có thể cung cấp, phát triển dịch vụ logistics toàn cầu, tin cậy dựa trên chất lượng dịch vụ, con người và công nghệ, cùng niềm đam mê với nghề ..."
1,https://www.topcv.vn/brand/educa/tuyen-dung/data-governance-specialist-j2055051.html,None,Data Governance Specialist,Data Analyst,20 - 30 triệu,Hà Nội,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Tầng 2-3-5-6 Tòa Comatce, 61 Ngụy Như Kon Tum, Phường Thanh Xuân (quận Thanh Xuân cũ)\n...",Thứ 2 - Thứ 6 (từ 08:00 đến 17:30) Thứ 7 (từ 08:00 đến 12:00)\nThứ 2 - Thứ 6 (từ 08:00 đến 17:30)\nThứ 7 (từ 08:00 đến 12:00),2 năm,Đại Học trở lên,Toàn thời gian,Nhân viên,1 người,"− Xây dựng, triển khai và duy trì khung Data Governance cho toàn hệ thống dữ liệu của Công ty. − Xây dựng chính sách, tiêu chuẩn, quy định về quản lý dữ liệu (data quality, data profiling, data se...","− Đã tốt nghiệp ĐH trở lên, ưu tiên các chuyên ngành về Công nghệ thông tin/ Hệ thống thông tin quản lý hoặc các lĩnh vực liên quan và tối thiểu 2 năm kinh nghiệm trong lĩnh vực Data Governance/ D...","Mức lương thỏa thuận theo năng lực. Thử việc hưởng 100% lương trong 2 tháng. Thưởng Lễ/Tết, thưởng hiệu quả làm việc: theo quy định của Công ty. Chính sách đánh giá điều chỉnh lương hàng năm minh ...",Chuyên môn Data Analyst,Còn 12 ngày để ứng tuyển,EDUPIA,https://edupia.vn/,NaN,500-1000,"Trụ sở: Tầng 2,3,5,6 - Tòa Vincem Comatce Tower, số 61 Ngụy Như Kon Tum, Thanh Xuân, Hà Nội. Chi nhánh: Tầng 6, Tòa nhà Báo Sinh Viên - Hoa Học Trò, Yên Hòa, Cầu Giấy, Hà Nội","Được thành lập năm 2018, Edupia là Edtech lớn nhất Việt Nam về Tiếng Anh Online cho trẻ em, đồng thời là Top 50 Edtech Nổi bật nhất Đông Nam Á năm 2022 & 2023. Qua nhiều năm phát triển, Edupia đến..."
2,https://www.topcv.vn/brand/educa/tuyen-dung/project-manager-du-an-ai-hub-j2067747.html,None,Project Manager Dự Án AI HUB,AI Engineer,30 - 35 triệu,Hà Nội,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Tầng 2,

## 1. Clean cơ bản

In [79]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 369 entries, 0 to 368
Data columns (total 24 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   job_url                  369 non-null    str   
 1   job_id                   0 non-null      object
 2   job_title_raw            369 non-null    str   
 3   job_slug_raw             369 non-null    str   
 4   salary_raw               369 non-null    str   
 5   location_raw             369 non-null    str   
 6   working_addresses_raw    369 non-null    str   
 7   working_times_raw        339 non-null    str   
 8   experience_raw           369 non-null    str   
 9   education_level_raw      369 non-null    str   
 10  employment_type_raw      369 non-null    str   
 11  job_level_raw            369 non-null    str   
 12  job_quantity_raw         369 non-null    str   
 13  description_raw          369 non-null    str   
 14  requirements_raw         369 non-null    str   
 15  

In [80]:
# Tạo các cột clean_* cho các cột text gốc
base_text_cols = [
    "job_title_raw",
    "job_slug_raw",
    "salary_raw",
    "location_raw",
    "working_addresses_raw",
    "working_times_raw",
    "experience_raw",
    "education_level_raw",
    "employment_type_raw",
    "job_level_raw",
    "job_quantity_raw",
    "description_raw",
    "requirements_raw",
    "benefits_raw",
    "tags_raw",
    "deadline_raw",
    "company_name_raw",
    "company_address_raw",
    "company_field_raw",
    "company_scale_raw",
    "company_description_raw",
]

for col in base_text_cols:
    if col in df_clean.columns:
        prefix = col.replace("_raw", "")
        df_clean[f"{prefix}_clean_light"] = df_clean[col].apply(clean_text_light)
        df_clean[f"{prefix}_clean_struct"] = df_clean[col].apply(clean_text_preserve_structure)
        df_clean[f"{prefix}_clean_strict"] = df_clean[col].apply(clean_text_strict)
        df_clean[f"{prefix}_clean_phobert"] = df_clean[col].apply(clean_text_for_phobert)

print("[INFO] Đã tạo xong các cột clean_*")
clean_cols = [c for c in df_clean.columns if "_clean_" in c]
print("Số cột clean:", len(clean_cols))
display(df_clean[[c for c in clean_cols[:12]]].head(2))

[INFO] Đã tạo xong các cột clean_*
Số cột clean: 84


,job_title_clean_light,job_title_clean_struct,job_title_clean_strict,job_title_clean_phobert,job_slug_clean_light,job_slug_clean_struct,job_slug_clean_strict,job_slug_clean_phobert,salary_clean_light,salary_clean_struct,salary_clean_strict,salary_clean_phobert
0,Junior FP&A Analyst - Hồ Chí Minh,Junior FP&A Analyst - Hồ Chí Minh,junior fp a analyst - hồ chí minh,Junior FP A Analyst - Hồ Chí Minh,Data Analyst,Data Analyst,data analyst,Data Analyst,12 - 20 triệu,12 - 20 triệu,12 - 20 triệu,12 - 20 triệu
1,Data Governance Specialist,Data Governance Specialist,data governance specialist,Data Governance Specialist,Data Analyst,Data Analyst,data analyst,Data Analyst,20 - 30 triệu,20 - 30 triệu,20 - 30 triệu,20 - 30 triệu


In [81]:
preview_cols = [
    # "job_title_raw", "job_title_clean_light", "job_title_clean_strict", "job_title_clean_phobert",
    # "job_slug_raw", "job_slug_clean_light", "job_slug_clean_strict", "job_slug_clean_phobert",
    # "requirements_raw", "requirements_clean_struct", "requirements_clean_strict", "requirements_clean_phobert",
    # "description_raw", "description_clean_struct","description_clean_light", "description_clean_phobert"
    "working_addresses_clean_light", "working_addresses_clean_struct", "working_addresses_clean_strict", "working_addresses_clean_phobert",
    "company_address_clean_light", "company_address_clean_struct", "company_address_clean_strict", "company_address_clean_phobert"
    
]
preview_cols = [c for c in preview_cols if c in df_clean.columns]

display(df_clean[preview_cols].head(3))

empty_ratio_df = pd.DataFrame({
    "column": preview_cols,
    "empty_ratio": [(df_clean[c].fillna("").str.strip() == "").mean() for c in preview_cols]
})
display(empty_ratio_df)

,working_addresses_clean_light,working_addresses_clean_struct,working_addresses_clean_strict,working_addresses_clean_phobert,company_address_clean_light,company_address_clean_struct,company_address_clean_strict,company_address_clean_phobert
0,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hồ Chí Minh: TTC 253 Hoàng Văn Thụ, Phường Tân Sơn Hòa (quận Tân Bình cũ)","(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hồ Chí Minh: TTC 253 Hoàng Văn Thụ, Phường Tân Sơn Hòa (quận Tân Bình cũ)",đã được cập nhật theo danh mục hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu - hồ chí minh ttc 253 hoàng văn thụ phường tân sơn hòa quận tân bình cũ,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hồ Chí Minh: TTC 253 Hoàng Văn Thụ, Phường Tân Sơn Hòa (quận Tân Bình cũ)","Addr: 11th Floor, TTC Tower, 253 Hoang Van Thu, Ward 2, Tan Binh District, Ho Chi Minh City, Viet Nam Addr: 10th Floor, 789 Tower, 147 Hoang Quoc Viet, Nghia Do Ward, Cau Giay District, Hanoi City...","Addr: 11th Floor, TTC Tower, 253 Hoang Van Thu, Ward 2, Tan Binh District, Ho Chi Minh City, Viet Nam Addr: 10th Floor, 789 Tower, 147 Hoang Quoc Viet, Nghia Do Ward, Cau Giay District, Hanoi City...",addr 11th floor ttc tower 253 hoang van thu ward 2 tan binh district ho chi minh city viet nam addr 10th floor 789 tower 147 hoang quoc viet nghia do ward cau giay district hanoi city viet nam. ad...,"Addr: 11th Floor, TTC Tower, 253 Hoang Van Thu, Ward 2, Tan Binh District, Ho Chi Minh City, Viet Nam Addr: 10th Floor, 789 Tower, 147 Hoang Quoc Viet, Nghia Do Ward, Cau Giay District, Hanoi City..."
1,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Tầng 2-3-5-6 Tòa Comatce, 61 Ngụy Như Kon Tum, Phường Thanh Xuân (quận Thanh Xuân cũ)\n...","(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Tầng 2-3-5-6 Tòa Comatce, 61 Ngụy Như Kon Tum, Phường Thanh Xuân (quận Thanh Xuân cũ)\n...",đã được cập nhật theo danh mục hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu - hà nội tầng 2-3-5-6 tòa comatce 61 ngụy như kon tum phường thanh xuân quận thanh xuân cũ hà nội,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Tầng 2-3-5-6 Tòa Comatce, 61 Ngụy Như Kon Tum, Phường Thanh Xuân (quận Thanh Xuân cũ)\n...","Trụ sở: Tầng 2,3,5,6 - Tòa Vincem Comatce Tower, số 61 Ngụy Như Kon Tum, Thanh Xuân, Hà Nội. Chi nhánh: Tầng 6, Tòa nhà Báo Sinh Viên - Hoa Học Trò, Yên Hòa, Cầu Giấy, Hà Nội","Trụ sở: Tầng 2,3,5,6 - Tòa Vincem Comatce Tower, số 61 Ngụy Như Kon Tum, Thanh Xuân, Hà Nội. Chi nhánh: Tầng 6, Tòa nhà Báo Sinh Viên - Hoa Học Trò, Yên Hòa, Cầu Giấy, Hà Nội",trụ sở tầng 2 3 5 6 - tòa vincem comatce tower số 61 ngụy như kon tum thanh xuân hà nội. chi nhánh tầng 6 tòa nhà báo sinh viên - hoa học trò yên hòa cầu giấy hà nội,"Trụ sở: Tầng 2,3,5,6 - Tòa Vincem Comatce Tower, số 61 Ngụy Như Kon Tum, Thanh Xuân, Hà Nội. Chi nhánh: Tầng 6, Tòa nhà Báo Sinh Viên - Hoa Học Trò, Yên Hòa, Cầu Giấy, Hà Nội"
2,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Tầng 2,3,5,6 Tòa Comatce Tower - 61 Ngụy Như Kon Tum, Thanh Xuân, Phường Thanh Xuân (qu...","(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Tầng 2,3,5,6 Tòa Comatce Tower - 61 Ngụy Như Kon Tum, Thanh Xuân, Phường Thanh Xuân (qu...",đã được cập nhật theo danh mục hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu - hà nội tầng 2 3 5 6 tòa comatce tower - 61 ngụy như kon tum thanh xuân phường thanh xuân quận than...,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Tầng 2,3,5,6 Tòa Comatce Tower - 61 N

,column,empty_ratio
0,working_addresses_clean_light,0.0
1,working_addresses_clean_struct,0.0
2,working_addresses_clean_strict,0.0
3,working_addresses_clean_phobert,0.0
4,company_address_clean_light,0.0
5,company_address_clean_struct,0.0
6,company_address_clean_strict,0.0
7,company_address_clean_phobert,0.0


In [82]:
# Tuỳ chọn hiển thị / debug
# pd.set_option("display.max_rows", 500)
# pd.set_option("display.max_columns", 200)
# pd.set_option("display.width", 200)
# pd.set_option("display.max_colwidth", 300)


### - Chuẩn hóa title

In [83]:
# Raw
unique_raw = df_clean["job_title_raw"].unique().tolist()
print("\n".join(unique_raw))   # mỗi giá trị 1 dòng
print("\n".join(unique_raw))

Junior FP&A Analyst - Hồ Chí Minh
Data Governance Specialist
Project Manager Dự Án AI HUB
AI Engineer
Data Analyst
Data Engineer
Fresher Data Engineer
On Job Training AI
Thực Tập Sinh Data Engineer
Data Analyst Teamleader (Collection Analytics)
Data Analyst Teamleader (Collection Analytics) - Thu Nhập 50 Triệu Tại Hồ Chí Minh
Chuyên Viên Kế Hoạch
Senior Agentic AI Expert
Fresher AI Computer Vision Engineer (Camera AI – DeepStream)
AI Platform Architect
Credit Risk Analytics And Modelling Expert
Expert, Fraud Risk Data Analytics And Portfolio Management
Data Analyst Workforce Management
CVCC Khoa Học Dữ Liệu - Hà Nội - TA174
CVCC Phân Tích Dữ Liệu (Lĩnh Vực Thu Hồi Nợ) - HN - TA105
Data Engineer - Hà Nội - TA105
Data Engineer - TA188
Data Scientist Expert - Hà Nội - TA150
Data Scientist - Hà Nội - TA105
Senior AI Engineer - Hà Nội - TA174
Admin Lên Đơn
Admin Sản Xuất
AI And Automation Intern
AI Artist
AI Automation Manager
AI Developer [ Hà Nội ] - Thu Nhập Hấp Dẫn Upto 50M
AI Developer

In [84]:
# Cắt phần đuôi thừa
TITLE_TRAILING_NOISE_PATTERNS = [
    # company / separator trailing noise
    r"\|\s*.*$",
    r"[_–—]+\s*.*$",  # trường hợp title nối đuôi bằng _ hoặc dash dài

    # compensation / benefits
    r"\-\s*lương.*$",
    r"\-\s*thu nhập.*$",
    r"\-\s*salary.*$",
    r"\-\s*upto.*$",
    r"\-\s*up to.*$",
    r"\-\s*đãi ngộ.*$",

    # location trailing
    r"\-\s*hà nội.*$",
    r"\-\s*hn.*$",
    r"\-\s*hồ chí minh.*$",
    r"\-\s*tp\.?\s*hcm.*$",
    r"\-\s*hcm.*$",
    r"\-\s*đà nẵng.*$",
    r"\-\s*tại hà nội.*$",
    r"\-\s*tại hồ chí minh.*$",
    r"\-\s*tại hcm.*$",

    # hiring urgency / employment mode
    r"\-\s*đi làm ngay.*$",
    r"\-\s*ob sớm.*$",
    r"\-\s*remote.*$",
    r"\-\s*hybrid.*$",
    r"\-\s*onsite.*$",

    # req code / internal code
    r"\-\s*ta\d+.*$",
    r"\-\s*ho\d{2}\.\d+.*$",
    r"\-\s*holt\.\d+.*$",

    # common extra recruitment phrases
    r"\-\s*từ \d+ năm kinh nghiệm.*$",
    r"\-\s*\d+\s*năm kinh nghiệm.*$",
    r"\-\s*không yêu cầu kinh nghiệm.*$",
    r"\-\s*thử việc.*$",
    r"\-\s*khối công nghệ thông tin.*$",
    r"\-\s*khối dữ liệu.*$",

    # standalone brackets noise if left behind in text
    r"\(\s*urgent\s*\)",
    r"\(\s*hot\s*\)",
    r"\(\s*remote\s*\)",
    r"\(\s*hybrid\s*\)",
    r"\(\s*onsite\s*\)",
    r"\(\s*english required\s*\)",
]
# Cắt bỏ nội dung trong ngoặc
BRACKET_NOISE_KEYWORDS = {
    # locations
    "hà nội", "ha noi", "hn",
    "hồ chí minh", "ho chi minh", "hcm", "tp hcm", "tphcm",
    "đà nẵng", "da nang",
    "quận 7", "quận 6", "mỹ đình", "nam từ liêm",

    # work modes / urgency
    "remote", "hybrid", "onsite", "urgent", "hot",
    "đi làm ngay", "ob sớm",
    "english required",

    # internal hiring codes
    "ta105", "ta150", "ta174", "ta188",
    "ho26.74", "ho26.76", "ho26.77", "ho26.78", "ho26.79", "ho26.84", "ho26.85",
    "holt.07", "holt.08", "holt.13", "holt.14", "holt.15", "holt.16", "holt.17",

    # common non-title bracket tags
    "game industry", "research web3", "domain erp", "domain edtech",
    "banking", "finance", "nhà máy", "collection analytics",
    "business machine learning", "loyalty/fintech",
}
# Chuẩn hóa các từ đồng nghĩa trong job title để dễ dàng mapping vào job family
TITLE_SYNONYM_MAP = {
    # core analytics / data
    "chuyên viên phân tích dữ liệu": "data analyst",
    "nhân viên phân tích dữ liệu": "data analyst",
    "cvcc phân tích dữ liệu": "data analyst",
    "cvcc khoa học dữ liệu": "data scientist",
    "chuyên viên dữ liệu": "data specialist",
    "chuyên gia dữ liệu": "data specialist",
    "chuyên viên kết nối phân tích dữ liệu": "data analyst",
    "chuyên viên xử lý dữ liệu": "data processing specialist",
    "nhân viên xử lý dữ liệu": "data processing specialist",
    "data analysis executive": "data analyst",
    "business data analyst": "business intelligence analyst",
    "business data analyst lead": "business intelligence analyst lead",
    "assistant manager data analyst": "data analyst manager",
    "retail data analyst supervisor": "data analyst supervisor",
    "data analyst workforce management": "workforce data analyst",
    "data analyst consultant": "data analyst",
    "data analyst teamleader": "data analyst lead",
    "trưởng nhóm data analyst": "data analyst lead",
    "trưởng phòng đào tạo data analyst": "data analyst manager",
    "senior data science & analysis": "senior data scientist",
    "hr data analysis": "hr data analyst",
    "ecommerce business data analyst": "business intelligence analyst",
    "phân tích dữ liệu kinh doanh": "business data analyst",

    # FP&A / finance analytics
    "fp&a analyst": "fp&a analyst",
    "finance planning & analysis associate": "fp&a analyst",
    "junior fp&a analyst": "fp&a analyst",
    "credit risk analytics and modelling expert": "risk analytics expert",
    "expert, fraud risk data analytics and portfolio management": "fraud risk analytics expert",
    "chuyên viên dữ liệu tài chính": "financial data analyst",
    "chuyên viên phân tích và quản lý dữ liệu tài chính": "financial data analyst",

    # business analyst / project / product
    "bi analyst": "business intelligence analyst",
    "business analyst": "business analyst",
    "chuyên viên business analyst": "business analyst",
    "chuyên viên phân tích nghiệp vụ": "business analyst",
    "chuyên viên phân tích hệ thống & nghiệp vụ": "business analyst",
    "chuyên viên phân tích nghiệp vụ mảng data": "data business analyst",
    "nhân viên phân tích dữ liệu ( business analyst )": "business analyst",
    "business analyst data": "data business analyst",
    "project manager dự án ai hub": "ai project manager",
    "quản lý dự án erp": "erp project manager",
    "chuyên viên quản trị dự án data warehouse": "data warehouse project manager",
    "project assistant": "project coordinator",
    "project assistant intern": "project coordinator intern",
    "project assistant - non-tech": "project coordinator",
    "trưởng nhóm phân tích kinh doanh": "business analysis lead",

    # data engineering / platform / database
    "data engineer": "data engineer",
    "chuyên viên dữ liệu ( data engineer )": "data engineer",
    "nhân viên data engineer": "data engineer",
    "kỹ sư dữ liệu": "data engineer",
    "kỹ sư dữ liệu lớn": "big data engineer",
    "big data admin": "big data engineer",
    "aws data engineer": "data engineer",
    "data engineer aws": "data engineer",
    "junior data integration engineer": "data integration engineer",
    "data platform engineer": "data platform engineer",
    "data platform operation": "data platform engineer",
    "database engineer": "database engineer",
    "database engineer ( dba )": "database administrator",
    "quản trị cơ sở dữ liệu": "database administrator",
    "database developer": "database developer",
    "reporting engineer power bi microsoft fabric": "bi engineer",
    "fresher data warehouse": "data warehouse engineer",
    "data management": "data management specialist",

    # data science / ml / ai
    "data scientist": "data scientist",
    "nhà khoa học dữ liệu": "data scientist",
    "chuyên viên phát triển khoa học dữ liệu": "data scientist",
    "chuyên viên mô hình và phân tích nâng cao": "data scientist",
    "chuyên viên cao cấp mô hình hóa và phân tích nâng cao": "senior data scientist",
    "chuyên viên, chuyên viên cao cấp khoa học dữ liệu": "data scientist",
    "ml engineer": "machine learning engineer",
    "machine learning engineer": "machine learning engineer",
    "ai engineer": "ai engineer",
    "kỹ sư ai": "ai engineer",
    "kỹ sư trí tuệ nhân tạo": "ai engineer",
    "chuyên viên trí tuệ nhân tạo": "ai engineer",
    "chuyên viên trí tuệ nhân tạo (ai)": "ai engineer",
    "chuyên viên ai": "ai engineer",
    "lập trình viên ai": "ai engineer",
    "nhân viên ai system engineer": "ai system engineer",
    "ai developer": "ai engineer",
    "ai generative engineer": "generative ai engineer",
    "ai engineering": "ai engineer",
    "ai - machine learning engineer": "machine learning engineer",
    "ai platform architect": "ai architect",
    "ai platform engineer": "ai platform engineer",
    "ai system engineer": "ai system engineer",
    "ai automation manager": "ai automation manager",
    "ai and automation intern": "ai automation intern",
    "product & ai automation intern": "ai automation intern",
    "intern ai solution engineer": "ai solutions engineer intern",
    "senior ai expert": "senior ai expert",
    "ai expert": "ai expert",
    "senior agentic ai expert": "senior ai expert",
    "nlp research engineer": "nlp engineer",
    "senior nlp engineer": "senior nlp engineer",
    "senior ai engineer / nlp engineer": "senior nlp engineer",
    "machine vision engineer": "computer vision engineer",
    "fresher ai computer vision engineer": "computer vision engineer",
    "senior ai research engineer": "ai research engineer",
    "ai research intern": "ai research intern",
    "ai quantitative researcher intern": "ai research intern",
    "software engineer (prompt engineering)": "prompt engineer",
    "ai artist": "ai creative specialist",
    "trưởng nhóm ai engineer": "ai engineer lead",
    "trưởng nhóm ai": "ai lead",
    "giám đốc trí tuệ nhân tạo (ai)": "ai director",
    "trưởng phòng công nghệ thị giác máy tính": "head of computer vision",
    "fresher ai (nlp)": "nlp engineer",
    "fresher ai": "ai engineer",

    # governance / quality / labeling / operations
    "data governance specialist": "data governance specialist",
    "data quality analyst": "data quality analyst",
    "data labeling specialist": "data labeling specialist",
    "thực tập sinh xử lý dữ liệu": "data processing intern",
    "điều phối dự án label data tiếng anh": "data labeling coordinator",
    "nhân viên gán nhãn dữ liệu tiếng nhật": "data labeling specialist",
    "nhân viên dán nhãn - tiếng hàn": "data labeling specialist",
    "nhân viên nhập liệu - xử lý dữ liệu": "data entry specialist",
    "nhân viên nhập và xử lý dữ liệu tiếng nhật": "data processing specialist",
    "nhân viên ngôn ngữ dữ liệu tiếng anh": "language data specialist",
    "nhân viên ngôn ngữ dữ liệu tiếng trung": "language data specialist",
    "nhân viên ngôn ngữ dữ liệu - tiếng pháp": "language data specialist",
    "nhân viên ngôn ngữ dữ liệu - tiếng tây ban nha /bồ đào nha/thái": "language data specialist",

    # internship / fresher / trainee canonical role names
    "on job training ai": "ai trainee",
    "mb trainee - ai engineer": "ai trainee",
    "thực tập sinh ai": "ai intern",
    "ai engineer intern": "ai intern",
    "thực tập sinh backend ai": "ai backend intern",
    "thực tập sinh computer vision ai": "computer vision intern",
    "thực tập sinh nghiên cứu & ứng dụng ai": "ai research intern",
    "thực tập sinh nội dung ai": "ai content intern",
    "data analyst intern": "data analyst intern",
    "intern data analyst": "data analyst intern",
    "thực tập sinh data analyst": "data analyst intern",
    "data science intern": "data science intern",
    "data engineer intern": "data engineer intern",
    "thực tập sinh data engineer": "data engineer intern",
    "fresher data engineer": "data engineer",
    "fresher data analytics engineer": "analytics engineer",
    "fresher data warehouse engineer": "data warehouse engineer",

    # misc but relevant to your current data
    "aiops specialist": "aiops specialist",
    "presale data and analytics": "data analytics presales",
    "power bi leader": "bi lead",
    "giảng viên power bi": "power bi trainer",
    "chuyên viên power bi,tableau": "bi analyst",
    "chuyên viên cao cấp kiến trúc giải pháp dữ liệu": "data architect",
    "data architect": "data architect",
    "gsd engineer - garment standard data engineer - kỹ sư hệ thống dữ liệu chuẩn": "data engineer",
    "tự động hoá dữ liệu": "data automation specialist",
}

In [85]:
# Quy tắc mapping job title vào job family
JOB_FAMILY_RULES = {
    "data_analytics": [
        "data analyst",
        "business intelligence analyst",
        "analytics engineer",
        "product analyst",
        "fp&a analyst",
        "business data analyst",
        "workforce data analyst",
        "financial data analyst",
        "risk analytics expert",
        "fraud risk analytics expert",
        "hr data analyst",
        "bi analyst",
        "bi engineer",
        "reporting analyst",
        "power bi",
        "tableau",
    ],
    "data_engineering": [
        "data engineer",
        "etl developer",
        "big data engineer",
        "data integration engineer",
        "data warehouse engineer",
        "data platform engineer",
        "database engineer",
        "database administrator",
        "database developer",
        "data architect",
        "aws data engineer",
        "big data",
        "kafka",
        "spark",
        "hadoop",
        "fabric",
    ],
    "data_science_ml": [
        "data scientist",
        "machine learning engineer",
        "ai engineer",
        "ml engineer",
        "nlp engineer",
        "computer vision engineer",
        "ai research engineer",
        "ai research intern",
        "generative ai engineer",
        "prompt engineer",
        "ai architect",
        "ai platform engineer",
        "ai system engineer",
        "aiops specialist",
        "machine vision engineer",
        "ai expert",
        "ai director",
        "head of computer vision",
    ],
    "data_governance_quality": [
        "data governance specialist",
        "data quality analyst",
        "data steward",
        "data management specialist",
        "data labeling specialist",
        "data labeling coordinator",
        "data processing specialist",
        "data processing intern",
        "data entry specialist",
        "language data specialist",
    ],
    "product_project_ba": [
        "business analyst",
        "data business analyst",
        "project manager",
        "ai project manager",
        "erp project manager",
        "project coordinator",
        "product manager",
        "product owner",
        "scrum master",
        "business analysis lead",
    ],
    "ai_automation_solutions": [
        "ai automation manager",
        "ai automation intern",
        "data automation specialist",
        "ai solutions engineer",
        "ai solutions engineer intern",
        "ai creative specialist",
        "ai content intern",
    ],
    "software_engineering": [
        "backend developer",
        "frontend developer",
        "fullstack developer",
        "software engineer",
        "backend engineer",
        "fullstack engineer",
        "developer fullstack",
        "dev ai fullstack",
    ],
    "operations_support": [
        "project coordinator",
        "operation planning specialist",
        "production data executive",
        "data specialist",
        "admin",
        "assistant",
        "coordinator",
    ],
}

# Từ khóa suy ra job family từ description_raw
JOB_FAMILY_DESCRIPTION_HINTS = {
    "data_analytics": [
        "dashboard", "report", "báo cáo", "phân tích dữ liệu", "business intelligence",
        "power bi", "tableau", "sql", "kpi", "insight"
    ],
    "data_engineering": [
        "etl", "pipeline", "data pipeline", "data warehouse", "dwh", "airflow",
        "spark", "kafka", "hadoop", "lakehouse", "fabric", "big data"
    ],
    "data_science_ml": [
        "machine learning", "deep learning", "nlp", "llm", "rag",
        "computer vision", "mô hình", "model training", "tensorflow", "pytorch",
        "generative ai", "genai"
    ],
    "data_governance_quality": [
        "data governance", "data quality", "master data", "data steward",
        "label", "labeling", "gán nhãn", "data cleaning", "data validation"
    ],
    "product_project_ba": [
        "business analyst", "phân tích nghiệp vụ", "user story", "brd", "frd",
        "stakeholder", "scrum", "agile", "project management", "quản lý dự án"
    ],
    "ai_automation_solutions": [
        "automation", "workflow automation", "solution design", "presales",
        "chatbot", "agent", "ai solution", "process automation"
    ],
    "software_engineering": [
        "backend", "frontend", "fullstack", "api", "microservice",
        "react", "nodejs", "java", "python development", "web development"
    ],
    "operations_support": [
        "vận hành", "operation", "điều phối", "coordinator", "assistant",
        "hỗ trợ", "nhập liệu", "back office"
    ],
}

# Xử lý nội dung trong ngoặc
def strip_bracket_noise(text):
    text = safe_str(text)
    if not text:
        return ""

    matches_round = re.findall(r"\((.*?)\)", text)
    for m in matches_round:
        normalized = clean_text_strict(m)
        if normalized in BRACKET_NOISE_KEYWORDS:
            text = text.replace(f"({m})", " ")

    matches_square = re.findall(r"\[(.*?)\]", text)
    for m in matches_square:
        normalized = clean_text_strict(m)
        if normalized in BRACKET_NOISE_KEYWORDS:
            text = text.replace(f"[{m}]", " ")

    return re.sub(r"\s+", " ", text).strip()

# Chuẩn hóa job title
def normalize_job_title(text):
    text = clean_text_light(text)
    if not text:
        return ""

    text = strip_bracket_noise(text)

    text = re.sub(r"\b(?:TA\d+|HO\d{2}\.\d+|HOLT\.\d+)\b", " ", text, flags=re.I)

    for pat in TITLE_TRAILING_NOISE_PATTERNS:
        text = re.sub(pat, " ", text, flags=re.I)

    text = re.sub(r"\s+", " ", text).strip(" -|_")
    lowered = text.lower().strip()

    if lowered in TITLE_SYNONYM_MAP:
        return TITLE_SYNONYM_MAP[lowered]

    return lowered

def infer_job_family_from_title(job_title_canonical):
    t = safe_str(job_title_canonical).lower()
    for family, keywords in JOB_FAMILY_RULES.items():
        if any(k in t for k in keywords):
            return family
    return "unknown"

def infer_job_family_from_description(text):
    t = clean_text_strict(text)
    if not t:
        return "unknown"

    scores = {}
    for family, keywords in JOB_FAMILY_DESCRIPTION_HINTS.items():
        score = 0
        for kw in keywords:
            if kw in t:
                score += 1
        scores[family] = score

    best_family = max(scores, key=scores.get)
    best_score = scores[best_family]

    return best_family if best_score > 0 else "unknown"

def resolve_job_family(title_family, desc_family):
    title_family = safe_str(title_family) or "unknown"
    desc_family = safe_str(desc_family) or "unknown"

    if title_family != "unknown" and desc_family == "unknown":
        return title_family, "title"

    if title_family == "unknown" and desc_family != "unknown":
        return desc_family, "description"

    if title_family != "unknown" and desc_family != "unknown":
        if title_family == desc_family:
            return title_family, "title+description"
        return title_family, "title_priority"

    return "unknown", "unknown"

In [86]:
df_clean["job_title_display"] = df_clean["job_title_clean_light"].fillna("")
df_clean["job_title_canonical"] = df_clean["job_title_raw"].apply(normalize_job_title)

df_clean["job_family_from_title"] = df_clean["job_title_canonical"].apply(infer_job_family_from_title)
df_clean["job_family_from_description"] = df_clean["description_clean_strict"].apply(infer_job_family_from_description)

family_resolved = [
    resolve_job_family(a, b)
    for a, b in zip(df_clean["job_family_from_title"], df_clean["job_family_from_description"])
]
df_clean["job_family"] = [x[0] for x in family_resolved]
df_clean["job_family_source"] = [x[1] for x in family_resolved]

df_clean["job_title_for_phobert"] = df_clean["job_title_display"].where(
    df_clean["job_title_display"].fillna("").str.strip() != "",
    df_clean["job_title_canonical"]
).apply(clean_text_for_phobert)

display(
    df_clean[
        [
            "job_title_raw",
            "job_title_display",
            "job_title_canonical",
            "job_family_from_title",
            "job_family_from_description",
            "job_family",
            "job_family_source",
            "job_title_for_phobert",
        ]
    ].head(10)
)

,job_title_raw,job_title_display,job_title_canonical,job_family_from_title,job_family_from_description,job_family,job_family_source,job_title_for_phobert
0,Junior FP&A Analyst - Hồ Chí Minh,Junior FP&A Analyst - Hồ Chí Minh,fp&a analyst,data_analytics,data_analytics,data_analytics,title+description,Junior FP A Analyst - Hồ Chí Minh
1,Data Governance Specialist,Data Governance Specialist,data governance specialist,data_governance_quality,data_governance_quality,data_governance_quality,title+description,Data Governance Specialist
2,Project Manager Dự Án AI HUB,Project Manager Dự Án AI HUB,ai project manager,product_project_ba,operations_support,product_project_ba,title_priority,Project Manager Dự Án AI HUB
3,AI Engineer,AI Engineer,ai engineer,data_science_ml,data_engineering,data_science_ml,title_priority,AI Engineer
4,AI Engineer,AI Engineer,ai engineer,data_science_ml,ai_automation_solutions,data_science_ml,title_priority,AI Engineer
5,Data Analyst,Data Analyst,data analyst,data_analytics,data_analytics,data_analytics,title+description,Data Analyst
6,Data Engineer,Data Engineer,data engineer,data_engineering,unknown,data_engineering,title,Data Engineer
7,Fresher Data Engineer,Fresher Data Engineer,data engineer,data_engineering,data_analytics,data_engineering,title_priority,Fresher Data Engineer
8,On Job Training AI,On Job Training AI,ai trainee,unknown,data_engineering,data_engineering,description,On Job Training AI
9,Thực Tập Sinh Data Engineer,Thực Tập Sinh Data Engineer,data engineer intern,data_engineering,data_engineering,data_engineering,title+description,Thực Tập Sinh Data Engineer


### - Chuẩn hóa địa điểm

In [87]:
VIETNAM_CITY_ALIASES = {
    "hà nội": "Hà Nội",
    "ha noi": "Hà Nội",
    "hn": "Hà Nội",
    "hồ chí minh": "Hồ Chí Minh",
    "ho chi minh": "Hồ Chí Minh",
    "tp hcm": "Hồ Chí Minh",
    "hcm": "Hồ Chí Minh",
    "sài gòn": "Hồ Chí Minh",
    "sai gon": "Hồ Chí Minh",
    "đà nẵng": "Đà Nẵng",
    "da nang": "Đà Nẵng",
    "hải phòng": "Hải Phòng",
    "hai phong": "Hải Phòng",
    "cần thơ": "Cần Thơ",
    "can tho": "Cần Thơ",
}

WORK_MODE_RULES = {
    "remote": [r"\bremote\b", r"làm việc từ xa", r"work from home", r"\bwfh\b"],
    "hybrid": [r"\bhybrid\b", r"linh hoạt", r"kết hợp onsite và remote"],
    "onsite": [r"\bonsite\b", r"tại văn phòng", r"làm việc tại công ty"],
}


def detect_city_from_text(text):
    t = clean_text_strict(text)
    for alias, canonical in VIETNAM_CITY_ALIASES.items():
        if alias in t:
            return canonical
    return None


def infer_work_mode(*texts):
    merged = " ".join([clean_text_strict(t) for t in texts if safe_str(t)])
    if not merged:
        return "unknown"
    for mode, patterns in WORK_MODE_RULES.items():
        for p in patterns:
            if re.search(p, merged, flags=re.I):
                return mode
    return "unknown"


def has_multi_location(text):
    t = clean_text_strict(text)
    hits = set()
    for alias, canonical in VIETNAM_CITY_ALIASES.items():
        if alias in t:
            hits.add(canonical)
    return len(hits) >= 2


def parse_working_address(raw_text):
    text = clean_text_light(raw_text)
    city = detect_city_from_text(text)
    district = None
    m = re.search(r"quận\s+(\d+|[a-zà-ỹ]+)", text, flags=re.I)
    if m:
        district = m.group(0).strip()

    return {
        "working_address_clean": text,
        "location_city": city,
        "location_district": district,
        "is_multi_location": has_multi_location(text)
    }


def normalize_location(location_raw, working_addresses_raw):
    city_1 = detect_city_from_text(location_raw)
    city_2 = detect_city_from_text(working_addresses_raw)
    return first_non_empty(city_1, city_2, clean_text_light(location_raw), clean_text_light(working_addresses_raw))


def parse_deadline(raw, reference_date=None):
    reference_date = reference_date or datetime.today().date()
    text = clean_text_light(raw)

    if not text:
        return {
            "deadline_clean": "",
            "deadline_date": None,
            "days_to_deadline": None,
            "deadline_type": "unknown",
            "is_expired": None,
        }

    # dd/mm/yyyy or dd-mm-yyyy
    m = re.search(r"(\d{1,2})[/-](\d{1,2})[/-](\d{2,4})", text)
    if m:
        day, month, year = map(int, m.groups())
        if year < 100:
            year += 2000
        try:
            dt = datetime(year, month, day).date()
            return {
                "deadline_clean": text,
                "deadline_date": str(dt),
                "days_to_deadline": (dt - reference_date).days,
                "deadline_type": "absolute_date",
                "is_expired": dt < reference_date,
            }
        except Exception:
            pass

    m = re.search(r"(\d+)\s*ngày", clean_text_strict(text))
    if m:
        days = int(m.group(1))
        dt = reference_date + timedelta(days=days)
        return {
            "deadline_clean": text,
            "deadline_date": str(dt),
            "days_to_deadline": days,
            "deadline_type": "relative_days",
            "is_expired": False,
        }

    return {
        "deadline_clean": text,
        "deadline_date": None,
        "days_to_deadline": None,
        "deadline_type": "unknown",
        "is_expired": None,
    }

In [88]:
address_parsed = df_clean["working_addresses_raw"].apply(parse_working_address)
address_df = pd.DataFrame(address_parsed.tolist(), index=df_clean.index)

for c in address_df.columns:
    df_clean[c] = address_df[c]

df_clean["location_norm"] = [
    normalize_location(a, b)
    for a, b in zip(df_clean["location_raw"], df_clean["working_addresses_raw"])
]

df_clean["work_mode"] = [
    infer_work_mode(a, b, c, d)
    for a, b, c, d in zip(
        df_clean["job_title_raw"],
        df_clean["location_raw"],
        df_clean["working_addresses_raw"],
        df_clean["description_raw"]
    )
]

deadline_parsed = df_clean["deadline_raw"].apply(parse_deadline)
deadline_df = pd.DataFrame(deadline_parsed.tolist(), index=df_clean.index)
for c in deadline_df.columns:
    df_clean[c] = deadline_df[c]

display(df_clean[
    [
        "location_raw", "working_addresses_raw", "location_city",
        "location_district", "location_norm", "work_mode",
        "deadline_raw", "deadline_date", "days_to_deadline", "deadline_type", "is_expired"
    ]
].head(10))

,location_raw,working_addresses_raw,location_city,location_district,location_norm,work_mode,deadline_raw,deadline_date,days_to_deadline,deadline_type,is_expired
0,Hồ Chí Minh (mới),"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hồ Chí Minh: TTC 253 Hoàng Văn Thụ, Phường Tân Sơn Hòa (quận Tân Bình cũ)",Hồ Chí Minh,quận Tân,Hồ Chí Minh,unknown,27/03/2026,2026-03-27,2,absolute_date,False
1,Hà Nội,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Tầng 2-3-5-6 Tòa Comatce, 61 Ngụy Như Kon Tum, Phường Thanh Xuân (quận Thanh Xuân cũ)\n...",Hà Nội,quận Thanh,Hà Nội,unknown,Còn 12 ngày để ứng tuyển,2026-04-06,12,relative_days,False
2,Hà Nội,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Tầng 2,3,5,6 Tòa Comatce Tower - 61 Ngụy Như Kon Tum, Thanh Xuân, Phường Thanh Xuân (qu...",Hà Nội,quận Thanh,Hà Nội,unknown,Còn 19 ngày để ứng tuyển,2026-04-13,19,relative_days,False
3,Hà Nội,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: FPT Tower, số 10 Phạm Văn Bạch, Phường Cầu Giấy (quận Cầu Giấy cũ)",Hà Nội,quận Cầu,Hà Nội,unknown,30/04/2026,2026-04-30,36,absolute_date,False
4,Hà Nội,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Landmark 72, Keangnam Phạm Hùng, Phường Yên Hòa (quận Cầu Giấy cũ)",Hà Nội,quận Cầu,Hà Nội,unknown,11/05/2026,2026-05-11,47,absolute_date,False
5,Hà Nội,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Keangnam Landmark 72, E6 Phạm Hùng, Phường Yên Hòa (quận Cầu Giấy cũ)",Hà Nội,quận Cầu,Hà Nội,unknown,08/04/2026,2026-04-08,14,absolute_date,False
6,Hà Nội,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Keangnam Landmark 72, Phường Yên Hòa (quận Cầu Giấy cũ)",Hà Nội,quận Cầu,Hà Nội,unknown,09/04/2026,2026-04-09,15,absolute_date,False
7,Hà Nội,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: Khu Công nghệ cao Hòa Lạc, Xã Hòa Lạc (huyện Thạch Thất cũ) - Hà Nội: FPT Tower, số 10 ...",Hà Nội,quận Cầu,Hà Nội,unknown,09/04/2026,2026-04-09,15,absolute_date,False
8,Hà Nội,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: FPT Tower, 10 Phạm Văn Bạch, Phường Cầu Giấy (quận Cầu Giấy cũ)",Hà Nội,quận Cầu,Hà Nội,unknown,30/04/2026,2026-04-30,36,absolute_date,False
9,Hà Nội,"(đã được cập nhật theo Danh mục Hành chính mới - thêm quận/huyện cũ tương ứng để dễ dàng tra cứu)\n- Hà Nội: F-Ville 3, Hòa Lạc, Xã Thạch Thất (huyện Thạch Thất cũ)",Hà Nội,NaN,Hà Nội,unknown,30/04/2026,2026-04-30,36,absolute_date,False


In [89]:
def clean_salary(raw):
    text = clean_text_light(raw).lower()
    text = text.replace("thoả", "thỏa")
    return text


def parse_numeric_token(num_text):
    s = safe_str(num_text).strip().replace(" ", "")
    if not s:
        return None

    if "," in s and "." in s:
        # chọn heuristic đơn giản
        s = s.replace(".", "").replace(",", ".")
    elif s.count(".") > 1:
        s = s.replace(".", "")
    elif s.count(",") > 1:
        s = s.replace(",", "")
    else:
        s = s.replace(",", ".")

    try:
        return float(s)
    except Exception:
        return None


def detect_salary_multiplier(text, currency):
    t = text.lower()
    if currency == "usd":
        return 1.0
    if "triệu" in t or "trieu" in t:
        return 1_000_000
    if "nghìn" in t or "ngan" in t or "k" in t:
        return 1_000
    return 1.0


def parse_salary_range(raw):
    text = clean_salary(raw)
    if not text:
        return {
            "salary_clean": "",
            "salary_min_value": None,
            "salary_max_value": None,
            "salary_currency": "unknown",
            "salary_period": "month",
            "salary_type": "unknown",
            "salary_is_negotiable": None,
            "salary_min_vnd_month": None,
            "salary_max_vnd_month": None,
        }

    currency = "usd" if "usd" in text or "$" in text else "vnd"
    negotiable = ("thỏa thuận" in text) or ("negotiable" in text)

    nums = re.findall(r"\d+(?:[.,]\d+)?", text)
    nums = [parse_numeric_token(x) for x in nums]
    nums = [x for x in nums if x is not None]

    multiplier = detect_salary_multiplier(text, currency)

    salary_min = None
    salary_max = None
    salary_type = "unknown"

    if negotiable and not nums:
        salary_type = "negotiable"
    elif "up to" in text or "tối đa" in text:
        if nums:
            salary_max = nums[0] * multiplier
            salary_type = "upper_bound"
    elif len(nums) >= 2:
        salary_min = nums[0] * multiplier
        salary_max = nums[1] * multiplier
        salary_type = "range"
    elif len(nums) == 1:
        salary_min = nums[0] * multiplier
        salary_max = nums[0] * multiplier
        salary_type = "fixed"

    if currency == "usd":
        # quy đổi tạm
        fx = 25000
        if salary_min is not None:
            salary_min = salary_min * fx
        if salary_max is not None:
            salary_max = salary_max * fx

    return {
        "salary_clean": text,
        "salary_min_value": salary_min,
        "salary_max_value": salary_max,
        "salary_currency": currency,
        "salary_period": "month",
        "salary_type": salary_type,
        "salary_is_negotiable": negotiable,
        "salary_min_vnd_month": salary_min,
        "salary_max_vnd_month": salary_max,
    }


salary_parsed = df_clean["salary_raw"].apply(parse_salary_range)
salary_df = pd.DataFrame(salary_parsed.tolist(), index=df_clean.index)
for c in salary_df.columns:
    df_clean[c] = salary_df[c]

display(df_clean[
    [
        "salary_raw", "salary_clean", "salary_min_value", "salary_max_value",
        "salary_currency", "salary_type", "salary_is_negotiable",
        "salary_min_vnd_month", "salary_max_vnd_month"
    ]
].head(10))

,salary_raw,salary_clean,salary_min_value,salary_max_value,salary_currency,salary_type,salary_is_negotiable,salary_min_vnd_month,salary_max_vnd_month
0,12 - 20 triệu,12 - 20 triệu,12000000.0,20000000.0,vnd,range,False,12000000.0,20000000.0
1,20 - 30 triệu,20 - 30 triệu,20000000.0,30000000.0,vnd,range,False,20000000.0,30000000.0
2,30 - 35 triệu,30 - 35 triệu,30000000.0,35000000.0,vnd,range,False,30000000.0,35000000.0
3,Thoả thuận,thỏa thuận,NaN,NaN,vnd,negotiable,True,NaN,NaN
4,Thoả thuận,thỏa thuận,NaN,NaN,vnd,negotiable,True,NaN,NaN
5,Thoả thuận,thỏa thuận,NaN,NaN,vnd,negotiable,True,NaN,NaN
6,Thoả thuận,thỏa thuận,NaN,NaN,vnd,negotiable,True,NaN,NaN
7,6 - 9 triệu,6 - 9 triệu,6000000.0,9000000.0,vnd,range,False,6000000.0,9000000.0
8,Thoả thuận,thỏa thuận,NaN,NaN,vnd,negotiable,True,NaN,NaN
9,Thoả thuận,thỏa thuận,NaN,NaN,vnd,negotiable,True,NaN,NaN


In [90]:
def clean_experience(text):
    return clean_text_strict(text)


def parse_experience_range(raw):
    text = clean_experience(raw)
    if not text:
        return {
            "experience_clean": "",
            "experience_min_years": None,
            "experience_max_years": None,
            "experience_type": "unknown",
        }

    if "không yêu cầu" in text or "no experience" in text:
        return {
            "experience_clean": text,
            "experience_min_years": 0.0,
            "experience_max_years": 0.0,
            "experience_type": "no_experience",
        }

    m_month = re.search(r"(\d+(?:\.\d+)?)\s*tháng", text)
    if m_month:
        months = float(m_month.group(1))
        years = months / 12.0
        return {
            "experience_clean": text,
            "experience_min_years": years,
            "experience_max_years": years,
            "experience_type": "fixed",
        }

    nums = re.findall(r"\d+(?:\.\d+)?", text)
    nums = [float(x) for x in nums] if nums else []

    if len(nums) >= 2:
        return {
            "experience_clean": text,
            "experience_min_years": nums[0],
            "experience_max_years": nums[1],
            "experience_type": "range",
        }

    if len(nums) == 1:
        val = nums[0]
        if "từ" in text or "+" in text or "trên" in text or "ít nhất" in text:
            return {
                "experience_clean": text,
                "experience_min_years": val,
                "experience_max_years": None,
                "experience_type": "minimum",
            }
        if "dưới" in text:
            return {
                "experience_clean": text,
                "experience_min_years": 0.0,
                "experience_max_years": val,
                "experience_type": "maximum",
            }
        return {
            "experience_clean": text,
            "experience_min_years": val,
            "experience_max_years": val,
            "experience_type": "fixed",
        }

    return {
        "experience_clean": text,
        "experience_min_years": None,
        "experience_max_years": None,
        "experience_type": "unknown",
    }


EDUCATION_MAP = {
    "phd": ["tiến sĩ", "phd", "doctor"],
    "master": ["thạc sĩ", "master"],
    "bachelor": ["đại học", "cử nhân", "bachelor"],
    "college": ["cao đẳng", "college"],
    "high_school": ["trung học", "high school"],
}

EMPLOYMENT_TYPE_MAP = {
    "full_time": ["toàn thời gian", "full time", "full-time"],
    "part_time": ["bán thời gian", "part time", "part-time"],
    "internship": ["thực tập", "internship", "intern"],
    "contract": ["hợp đồng", "contract"],
    "freelance": ["freelance", "cộng tác viên"],
    "temporary": ["temporary", "thời vụ"],
}


def normalize_education_level(text):
    t = clean_text_strict(text)
    for level, kws in EDUCATION_MAP.items():
        if any(k in t for k in kws):
            return level
    return "unknown"


def normalize_employment_type(text):
    t = clean_text_strict(text)
    for level, kws in EMPLOYMENT_TYPE_MAP.items():
        if any(k in t for k in kws):
            return level
    return "unknown"


JOB_LEVEL_RULES = {
    "director": [r"\bdirector\b", r"\bhead\b", r"giám đốc", r"trưởng phòng"],
    "manager": [r"\bmanager\b", r"quản lý", r"trưởng bộ phận"],
    "lead": [r"\blead\b", r"team lead", r"leader", r"trưởng nhóm"],
    "senior": [r"\bsenior\b", r"\bsr\b", r"cao cấp"],
    "middle": [r"\bmiddle\b", r"\bmid\b"],
    "junior": [r"\bjunior\b", r"\bjr\b"],
    "fresher": [r"\bfresher\b", r"trainee", r"tập sự"],
    "intern": [r"\bintern\b", r"thực tập"],
}


def normalize_job_level(text):
    t = clean_text_strict(text)
    if not t:
        return "unknown"

    for lvl, patterns in JOB_LEVEL_RULES.items():
        for p in patterns:
            if re.search(p, t, flags=re.I):
                return lvl
    return "unknown"


exp_parsed = df_clean["experience_raw"].apply(parse_experience_range)
exp_df = pd.DataFrame(exp_parsed.tolist(), index=df_clean.index)
for c in exp_df.columns:
    df_clean[c] = exp_df[c]

df_clean["education_level_norm"] = df_clean["education_level_raw"].apply(normalize_education_level)
df_clean["employment_type_norm"] = df_clean["employment_type_raw"].apply(normalize_employment_type)
df_clean["job_level_norm"] = df_clean["job_level_raw"].apply(normalize_job_level)

df_clean["seniority_final"] = df_clean["job_level_norm"].fillna("unknown")
df_clean["seniority_source"] = np.where(
    df_clean["seniority_final"].fillna("unknown") != "unknown",
    "job_level_raw",
    "unknown"
)

display(df_clean[
    [
        "experience_raw", "experience_min_years", "experience_max_years", "experience_type",
        "education_level_raw", "education_level_norm",
        "employment_type_raw", "employment_type_norm",
        "job_level_raw", "job_level_norm",
        "seniority_final", "seniority_source"
    ]
].head(10))

,experience_raw,experience_min_years,experience_max_years,experience_type,education_level_raw,education_level_norm,employment_type_raw,employment_type_norm,job_level_raw,job_level_norm,seniority_final,seniority_source
0,2 năm,2.0,2.0,fixed,Đại Học trở lên,bachelor,Toàn thời gian,full_time,Nhân viên,unknown,unknown,unknown
1,2 năm,2.0,2.0,fixed,Đại Học trở lên,bachelor,Toàn thời gian,full_time,Nhân viên,unknown,unknown,unknown
2,2 năm,2.0,2.0,fixed,Đại Học trở lên,bachelor,Toàn thời gian,full_time,Nhân viên,unknown,unknown,unknown
3,3 năm,3.0,3.0,fixed,Đại Học trở lên,bachelor,Toàn thời gian,full_time,Nhân viên,unknown,unknown,unknown
4,2 năm,2.0,2.0,fixed,Đại Học trở lên,bachelor,Toàn thời gian,full_time,Nhân viên,unknown,unknown,unknown
5,3 năm,3.0,3.0,fixed,Đại Học trở lên,bachelor,Toàn thời gian,full_time,Nhân viên,unknown,unknown,unknown
6,3 năm,3.0,3.0,fixed,Đại Học trở lên,bachelor,Toàn thời gian,full_time,Nhân viên,unknown,unknown,unknown
7,Không yêu cầu,0.0,0.0,no_experience,Đại Học trở lên,bachelor,Toàn thời gian,full_time,Trưởng nhóm,lead,lead,job_level_raw
8,Không yêu cầu,0.0,0.0,no_experience,Đại Học trở lên,bachelor,Toàn thời gian,full_time,Trưởng nhóm,lead,lead,job_level_raw
9,Không yêu cầu,0.0,0.0,no_experience,Đại Học trở lên,bachelor,Toàn thời gian,full_time,Trưởng nhóm,lead,lead,job_level_raw


In [91]:
quality_summary = pd.DataFrame({
    "metric": [
        "job_family_unknown_rate",
        "job_family_from_title_unknown_rate",
        "job_family_from_description_unknown_rate",
        "seniority_final_unknown_rate",
    ],
    "value": [
        (df_clean["job_family"] == "unknown").mean(),
        (df_clean["job_family_from_title"] == "unknown").mean(),
        (df_clean["job_family_from_description"] == "unknown").mean(),
        (df_clean["seniority_final"] == "unknown").mean(),
    ]
})

display(quality_summary)

display(
    df_clean[
        [
            "job_title_raw",
            "job_title_canonical",
            "description_raw",
            "job_family_from_title",
            "job_family_from_description",
            "job_family",
            "job_family_source",
            "job_level_raw",
            "job_level_norm",
            "seniority_final",
        ]
    ].sample(min(10, len(df_clean)), random_state=42)
)

,metric,value
0,job_family_unknown_rate,0.029810
1,job_family_from_title_unknown_rate,0.300813
2,job_family_from_description_unknown_rate,0.056911
3,seniority_final_unknown_rate,0.620596


,job_title_raw,job_title_canonical,description_raw,job_family_from_title,job_family_from_description,job_family,job_family_source,job_level_raw,job_level_norm,seniority_final
326,Senior Data Engineer (Loyalty/Fintech),senior data engineer,"Mục tiêu vị trí\nĐảm bảo hoạt động trơn tru của các hệ thống và dịch vụ dữ liệu của Data Services\nXây dựng và phát triển hạ tầng Dữ liệu của Data Services phục vụ các hoạt động quản trị, vận hành...",data_engineering,data_engineering,data_engineering,title+description,Nhân viên,unknown,unknown
33,AI Developer [ Hà Nội ] - Thu Nhập Hấp Dẫn Upto 50M,ai engineer,"Nghiên cứu, phát triển và tối ưu các hệ thống\nOCR (Optical Character Recognition)\ncho:\nVăn bản in, chữ viết tay\nTiếng Việt, tiếng Nhật, tiếng Anh (đa ngôn ngữ)\nXây dựng pipeline\nDocument Und...",data_science_ml,data_science_ml,data_science_ml,title+description,Nhân viên,unknown,unknown
15,Fresher AI Computer Vision Engineer (Camera AI – DeepStream),fresher ai computer vision engineer (camera ai - deepstream),DXTech- công ty con Softdreams chuyên về mảng AI đang tìm kiếm Fresher AI Computer Vision Engineer tham gia phát triển các hệ thống Camera AI thông minh phục vụ cho: • Trường học (nhận diện học si...,data_science_ml,data_science_ml,data_science_ml,title+description,Nhân viên,unknown,unknown
345,Thực Tập Sinh Nội Dung AI (AI Content Intern),thực tập sinh nội dung ai (ai content intern),"I. Sáng tạo & Hỗ trợ phát triển nội dung\nHỗ trợ team lên ý tưởng video TikTok theo định hướng nội dung của hệ thống nhà hàng.\nNghiên cứu trend, format mới và đề xuất ý tưởng phù hợp với ngành F&...",ai_automation_solutions,operations_support,ai_automation_solutions,title_priority,Thực tập sinh,intern,intern
57,AI Engineer,ai engineer,"I. MỤC ĐÍCH VỊ TRÍ\nVị trí AI Engineer / GenAI Engineer chịu trách nhiệm nghiên cứu, phát triển và triển khai các giải pháp Trí tuệ nhân tạo (AI) nhằm nâng cao hiệu quả vận hành và chất lượng sản ...",data_science_ml,data_science_ml,data_science_ml,title+description,Trưởng nhóm,lead,lead
239,Junior/Middle Data Analyst Mảng App Mobile,junior/middle data analyst mảng app mobile,"Xây dựng hệ thống KPI phục vụ theo dõi hiệu quả hoạt động kinh doanh\nKhai thác và phân tích dữ liệu từ kho dữ liệu để tìm ra xu hướng.\nLàm việc với các phòng ban nhằm xác định nhu cầu dữ liệu, đ...",data_analytics,data_analytics,data_analytics,title+description,Nhân viên,unknown,unknown
76,AI Research Intern,ai research intern,"1/ Research & Discovery\n- Nghiên cứu các AI / GenAI use cases trong các lĩnh vực: Retail, E-commerce, Marketing, Sales, Operations, Supply Chain, HR, Finance\n- Theo dõi xu hướng AI mới: LLMs, AI...",data_science_ml,data_science_ml,data_science_ml,title+description,Trưởng nhóm,lead,lead
119,Chuyên Viên Phân Tích Nghiệp Vụ Mảng Data,data business analyst,"Làm rõ, đánh giá và chuyển đổi các yêu cầu kinh doanh thành các yêu cầu kỹ thuật dữ liệu theo các mẫu đã định sẵn\nPhối hợp với Kiến trúc sư dữ liệu để xác định và xác thực ánh xạ dữ liệu.\nHỗ trợ...",product_project_ba,data_science_ml,product_project_ba,title_priority,Nhân viên,unknown,unknown
305,Presale Data And Analytics ( Banking ),data analytics presales,"Solution Consulting & Pre-Sales\n- Engage in pre-sales activities, solution consulting, and solution presentations/workshops for customers and partners.\n- Conduct solution demos and Proof of Conc...",unknown,unknown,unknown,unknown,Nhân viên,unknown,unknown
126,Chuyên Viên Phát Triển Khoa Học Dữ Liệu - Data Scientist - Khối Công Nghệ Thông Tin (HO26.84),chuyên viên phát triển khoa học dữ liệu - data scientist,"Phân tích và khai thác dữ liệu: Thu thập, làm sạch và xử lý dữ liệu từ nhiều nguồn khác nhau để phục vụ phân tích.\nPhát hiện xu hướng và đưa ra dự báo: Sử dụng các phương pháp phân tích để nhận d...",data_science_ml,operations_support,data_science_ml,title_priority,Nhân viên,unknown,unknown


In [92]:
def normalize_tags(text):
    text = clean_text_light(text)
    if not text:
        return []
    parts = re.split(r"[,\|;/\n]+", text)
    parts = [clean_text_strict(p) for p in parts]
    parts = [p for p in parts if p]
    return deduplicate_list(parts)


df_clean["tags_list"] = df_clean["tags_raw"].apply(normalize_tags)
df_clean["tags_text_phobert"] = df_clean["tags_list"].apply(
    lambda xs: ", ".join(xs) if isinstance(xs, list) else ""
).apply(clean_text_for_phobert)

display(df_clean[["tags_raw", "tags_list", "tags_text_phobert"]].head(10))

,tags_raw,tags_list,tags_text_phobert
0,Chuyên môn Data Analyst; Tài chính; Kế toán,"[chuyên môn data analyst, tài chính, kế toán]","chuyên môn data analyst, tài chính, kế toán"
1,Chuyên môn Data Analyst,[chuyên môn data analyst],chuyên môn data analyst
2,Chuyên môn IT Project Manager,[chuyên môn it project manager],chuyên môn it project manager
3,Chuyên môn AI Engineer; IT - Phần mềm,"[chuyên môn ai engineer, it - phần mềm]","chuyên môn ai engineer, it - phần mềm"
4,Chuyên môn AI Engineer,[chuyên môn ai engineer],chuyên môn ai engineer
5,Chuyên môn Data Analyst,[chuyên môn data analyst],chuyên môn data analyst
6,Chuyên môn Data Engineer,[chuyên môn data engineer],chuyên môn data engineer
7,Chuyên môn Data Engineer; IT - Phần mềm,"[chuyên môn data engineer, it - phần mềm]","chuyên môn data engineer, it - phần mềm"
8,Chuyên môn AI Engineer,[chuyên môn ai engineer],chuyên môn ai engineer
9,Chuyên môn Data Engineer,[chuyên môn data engineer],chuyên môn data engineer


In [93]:
SKILL_TAXONOMY = {
    "python": {"aliases": ["python"], "group": "programming"},
    "sql": {"aliases": ["sql", "postgresql", "mysql", "sql server"], "group": "database"},
    "excel": {"aliases": ["excel", "microsoft excel"], "group": "analytics_tools"},
    "power bi": {"aliases": ["power bi", "powerbi"], "group": "bi_tools"},
    "tableau": {"aliases": ["tableau"], "group": "bi_tools"},
    "pandas": {"aliases": ["pandas"], "group": "python_libs"},
    "numpy": {"aliases": ["numpy"], "group": "python_libs"},
    "scikit-learn": {"aliases": ["scikit-learn", "sklearn"], "group": "ml_libs"},
    "tensorflow": {"aliases": ["tensorflow"], "group": "ml_libs"},
    "pytorch": {"aliases": ["pytorch", "torch"], "group": "ml_libs"},
    "machine learning": {"aliases": ["machine learning", "ml"], "group": "ml_concepts"},
    "deep learning": {"aliases": ["deep learning", "dl"], "group": "ml_concepts"},
    "statistics": {"aliases": ["statistics", "thống kê"], "group": "analytics_concepts"},
    "etl": {"aliases": ["etl"], "group": "data_engineering"},
    "airflow": {"aliases": ["airflow", "apache airflow"], "group": "data_engineering"},
    "spark": {"aliases": ["spark", "apache spark", "pyspark"], "group": "data_engineering"},
    "hadoop": {"aliases": ["hadoop"], "group": "data_engineering"},
    "kafka": {"aliases": ["kafka", "apache kafka"], "group": "data_engineering"},
    "aws": {"aliases": ["aws", "amazon web services"], "group": "cloud"},
    "gcp": {"aliases": ["gcp", "google cloud platform"], "group": "cloud"},
    "azure": {"aliases": ["azure"], "group": "cloud"},
    "docker": {"aliases": ["docker"], "group": "devops"},
    "kubernetes": {"aliases": ["kubernetes", "k8s"], "group": "devops"},
    "git": {"aliases": ["git", "github", "gitlab"], "group": "dev_tools"},
    "api": {"aliases": ["api", "rest api", "restful api"], "group": "backend"},
    "fastapi": {"aliases": ["fastapi"], "group": "backend"},
    "flask": {"aliases": ["flask"], "group": "backend"},
    "llm": {"aliases": ["llm", "large language model", "large language models"], "group": "ai"},
    "nlp": {"aliases": ["nlp", "natural language processing"], "group": "ai"},
    "computer vision": {"aliases": ["computer vision", "cv"], "group": "ai"},
    "data modeling": {"aliases": ["data modeling", "data model"], "group": "data_modeling"},
    "data warehouse": {"aliases": ["data warehouse", "dwh"], "group": "data_modeling"},
    "bigquery": {"aliases": ["bigquery"], "group": "database"},
    "snowflake": {"aliases": ["snowflake"], "group": "database"},
    "oracle": {"aliases": ["oracle"], "group": "database"},
    "sas": {"aliases": ["sas"], "group": "analytics_tools"},
    "r": {"aliases": [" r ", "(r)", "ngôn ngữ r"], "group": "programming"},
    "communication": {"aliases": ["communication", "giao tiếp"], "group": "soft_skills"},
    "problem solving": {"aliases": ["problem solving", "giải quyết vấn đề"], "group": "soft_skills"},
}

In [94]:
REQUIRED_HINTS = [
    "bắt buộc", "required", "must have", "cần có", "thành thạo", "proficient", "kinh nghiệm với"
]
PREFERRED_HINTS = [
    "ưu tiên", "preferred", "nice to have", "là lợi thế", "plus point"
]


def alias_to_regex(alias):
    alias = safe_str(alias)
    if not alias:
        return None

    alias_strict = alias.strip()
    if alias_strict.lower() == "r":
        return r"(?<!\w)r(?!\w)"
    if alias_strict.lower() == "ml":
        return r"(?<!\w)ml(?!\w)"
    if alias_strict.lower() == "cv":
        return r"(?<!\w)cv(?!\w)"
    return r"(?<!\w)" + re.escape(alias_strict) + r"(?!\w)"


SKILL_PATTERNS = {}
for skill, meta in SKILL_TAXONOMY.items():
    pats = []
    for alias in meta["aliases"]:
        pat = alias_to_regex(alias)
        if pat:
            pats.append(re.compile(pat, flags=re.I))
    SKILL_PATTERNS[skill] = pats


def infer_skill_importance(segment, source_field):
    s = clean_text_strict(segment)
    if any(h in s for h in PREFERRED_HINTS):
        return "preferred"
    if source_field == "requirements":
        return "required"
    if any(h in s for h in REQUIRED_HINTS):
        return "required"
    return "mentioned"


def extract_skill_records_from_text(text, source_field="unknown"):
    text = clean_text_preserve_structure(text)
    if not text:
        return []

    segments = [seg.strip() for seg in re.split(r"[\n•\-;]+", text) if seg.strip()]
    records = []

    for seg in segments:
        for skill, patterns in SKILL_PATTERNS.items():
            matched = False
            for p in patterns:
                if p.search(seg):
                    matched = True
                    break
            if matched:
                records.append({
                    "skill": skill,
                    "skill_group": SKILL_TAXONOMY[skill]["group"],
                    "source_field": source_field,
                    "importance": infer_skill_importance(seg, source_field),
                    "excerpt": seg[:300]
                })

    # dedup
    seen = set()
    out = []
    for r in records:
        key = (r["skill"], r["source_field"], r["importance"])
        if key not in seen:
            seen.add(key)
            out.append(r)
    return out


def merge_skill_records(*record_lists):
    merged = []
    seen = set()
    for records in record_lists:
        for r in records:
            key = (r["skill"], r["source_field"], r["importance"])
            if key not in seen:
                seen.add(key)
                merged.append(r)
    return merged


def list_from_records(records, key, importance_filter=None):
    vals = []
    for r in records:
        if importance_filter and r.get("importance") != importance_filter:
            continue
        if r.get(key):
            vals.append(r[key])
    return deduplicate_list(vals)

In [95]:
title_skill_records = df_clean["job_title_clean_phobert"].apply(lambda x: extract_skill_records_from_text(x, "title"))
tag_skill_records = df_clean["tags_text_phobert"].apply(lambda x: extract_skill_records_from_text(x, "tags"))
req_skill_records = df_clean["requirements_clean_phobert"].apply(lambda x: extract_skill_records_from_text(x, "requirements"))
desc_skill_records = df_clean["description_clean_phobert"].apply(lambda x: extract_skill_records_from_text(x, "description"))

df_clean["skill_records"] = [
    merge_skill_records(a, b, c, d)
    for a, b, c, d in zip(title_skill_records, tag_skill_records, req_skill_records, desc_skill_records)
]

df_clean["skills_extracted"] = df_clean["skill_records"].apply(lambda rs: list_from_records(rs, "skill"))
df_clean["skills_required"] = df_clean["skill_records"].apply(lambda rs: list_from_records(rs, "skill", "required"))
df_clean["skills_preferred"] = df_clean["skill_records"].apply(lambda rs: list_from_records(rs, "skill", "preferred"))
df_clean["skill_groups"] = df_clean["skill_records"].apply(lambda rs: list_from_records(rs, "skill_group"))

df_clean["skills_text_phobert"] = df_clean["skills_extracted"].apply(
    lambda xs: ", ".join(xs) if isinstance(xs, list) else ""
)
df_clean["skills_required_text_phobert"] = df_clean["skills_required"].apply(
    lambda xs: ", ".join(xs) if isinstance(xs, list) else ""
)
df_clean["skills_preferred_text_phobert"] = df_clean["skills_preferred"].apply(
    lambda xs: ", ".join(xs) if isinstance(xs, list) else ""
)

display(df_clean[
    [
        "job_title_raw", "skills_extracted", "skills_required",
        "skills_preferred", "skill_groups",
        "skills_text_phobert", "skills_required_text_phobert"
    ]
].head(10))

,job_title_raw,skills_extracted,skills_required,skills_preferred,skill_groups,skills_text_phobert,skills_required_text_phobert
0,Junior FP&A Analyst - Hồ Chí Minh,[communication],[communication],[],[soft_skills],communication,communication
1,Data Governance Specialist,"[sql, etl, data warehouse]",[],"[sql, etl, data warehouse]","[database, data_engineering, data_modeling]","sql, etl, data warehouse",
2,Project Manager Dự Án AI HUB,[communication],[communication],[],[soft_skills],communication,communication
3,AI Engineer,"[python, machine learning, llm]","[machine learning, python, llm]","[python, machine learning, llm]","[programming, ml_concepts, ai]","python, machine learning, llm","machine learning, python, llm"
4,AI Engineer,"[llm, api]",[llm],[],"[ai, backend]","llm, api",llm
5,Data Analyst,"[machine learning, etl, data warehouse, oracle, problem solving]","[machine learning, etl, data warehouse, oracle, problem solving]",[],"[ml_concepts, data_engineering, data_modeling, database, soft_skills]","machine learning, etl, data warehouse, oracle, problem solving","machine learning, etl, data warehouse, oracle, problem solving"
6,Data Engineer,"[etl, sql, oracle, git]","[etl, sql, oracle, git]",[],"[data_engineering, database, dev_tools]","etl, sql, oracle, git","etl, sql, oracle, git"
7,Fresher Data Engineer,"[python, sql, etl, spark, communication, problem solving, aws, power bi]","[communication, problem solving, sql, etl, spark]","[python, sql, etl, spark]","[programming, database, data_engineering, soft_skills, cloud, bi_tools]","python, sql, etl, spark, communication, problem solving, aws, power bi","communication, problem solving, sql, etl, spark"
8,On Job Training AI,"[communication, python, problem solving, aws, azure, machine learning, deep learning]","[communication, python, aws, azure]","[python, problem solving]","[soft_skills, programming, cloud, ml_concepts]","communication, python, problem solving, aws, azure, machine learning, deep learning","communication, python, aws, azure"
9,Thực Tập Sinh Data Engineer,"[etl, airflow, docker, data modeling, machine learning]","[etl, airflow, docker, data modeling]",[],"[data_engineering, devops, data_modeling, ml_concepts]","etl, airflow, docker, data modeling, machine learning","etl, airflow, docker, data modeling"


In [96]:
job_skill_rows = []
for _, row in df_clean.iterrows():
    for r in row["skill_records"]:
        job_skill_rows.append({
            "job_url": row.get("job_url"),
            "job_title_display": row.get("job_title_display"),
            "job_title_canonical": row.get("job_title_canonical"),
            "job_family": row.get("job_family"),
            "location_norm": row.get("location_norm"),
            "skill": r["skill"],
            "skill_group": r["skill_group"],
            "source_field": r["source_field"],
            "importance": r["importance"],
            "excerpt": r["excerpt"],
        })

job_skill_map_df = pd.DataFrame(job_skill_rows)
display(job_skill_map_df.head(10))

if len(job_skill_map_df) > 0:
    role_skill_stats_df = (
        job_skill_map_df.groupby(["job_family", "skill", "importance"])
        .size()
        .reset_index(name="job_count")
        .sort_values(["job_family", "job_count"], ascending=[True, False])
    )
else:
    role_skill_stats_df = pd.DataFrame(columns=["job_family", "skill", "importance", "job_count"])

display(role_skill_stats_df.head(20))
print("Tỷ lệ job không extract được skill:", (df_clean["skills_extracted"].apply(len) == 0).mean())

,job_url,job_title_display,job_title_canonical,job_family,location_norm,skill,skill_group,source_field,importance,excerpt
0,https://www.topcv.vn/brand/beeogisticscorporation/tuyen-dung/junior-fpa-analyst-ho-chi-minh-j1851113.html,Junior FP&A Analyst - Hồ Chí Minh,fp&a analyst,data_analytics,Hồ Chí Minh,communication,soft_skills,requirements,required,"Interpretation and analysis skills, diligence, and honesty in communication."
1,https://www.topcv.vn/brand/educa/tuyen-dung/data-governance-specialist-j2055051.html,Data Governance Specialist,data governance specialist,data_governance_quality,Hà Nội,sql,database,requirements,preferred,"Thành thạo kĩ năng SQL, có tư duy triển khai Data Warehouse, ETL, BI là lợi thế."
2,https://www.topcv.vn/brand/educa/tuyen-dung/data-governance-specialist-j2055051.html,Data Governance Specialist,data governance specialist,data_governance_quality,Hà Nội,etl,data_engineering,requirements,preferred,"Thành thạo kĩ năng SQL, có tư duy triển khai Data Warehouse, ETL, BI là lợi thế."
3,https://www.topcv.vn/brand/educa/tuyen-dung/data-governance-specialist-j2055051.html,Data Governance Specialist,data governance specialist,data_governance_quality,Hà Nội,data warehouse,data_modeling,requirements,preferred,"Thành thạo kĩ năng SQL, có tư duy triển khai Data Warehouse, ETL, BI là lợi thế."
4,https://www.topcv.vn/brand/educa/tuyen-dung/data-governance-specialist-j2055051.html,Data Governance Specialist,data governance specialist,data_governance_quality,Hà Nội,data warehouse,data_modeling,description,mentioned,Tham gia thiết kế mô hình quản trị dữ liệu cho các hệ thống Data Warehouse/ BI.
5,https://www.topcv.vn/brand/educa/tuyen-dung/project-manager-du-an-ai-hub-j2067747.html,Project Manager Dự Án AI HUB,ai project manager,product_project_ba,Hà Nội,communication,soft_skills,requirements,required,"driven): Có khả năng đọc hiểu và phân tích dữ liệu để đưa ra quyết định. Kỹ năng giao tiếp và làm việc liên phòng ban tốt. Khả năng học nhanh và thích nghi với các công nghệ mới. Nghiêm túc, chỉn ..."
6,https://www.topcv.vn/brand/fptis/tuyen-dung/ai-engineer-j2026003.html,AI Engineer,ai engineer,data_science_ml,Hà Nội,python,programming,requirements,preferred,"Kinh nghiệm AI/ML ứng dụng thực tế Python và xử lý dữ liệu tốt Tư duy production và tối ưu chi phí Ưu tiên ứng viên đã có kinh nghiệm ở các vị trí tương tự. Có kỹ năng làm việc nhóm, quản lý thời ..."
7,https://www.topcv.vn/brand/fptis/tuyen-dung/ai-engineer-j2026003.html,AI Engineer,ai engineer,data_science_ml,Hà Nội,machine learning,ml_concepts,requirements,preferred,"Kinh nghiệm AI/ML ứng dụng thực tế Python và xử lý dữ liệu tốt Tư duy production và tối ưu chi phí Ưu tiên ứng viên đã có kinh nghiệm ở các vị trí tương tự. Có kỹ năng làm việc nhóm, quản lý thời ..."
8,https://www.topcv.vn/brand/fptis/tuyen-dung/ai-engineer-j2026003.html,AI Engineer,ai engineer,data_science_ml,Hà Nội,llm,ai,requirements,preferred,"Kinh nghiệm AI/ML ứng dụng thực tế Python và xử lý dữ liệu tốt Tư duy production và tối ưu chi phí Ưu tiên ứng viên đã có kinh nghiệm ở các vị trí tương tự. Có kỹ năng làm việc nhóm, quản lý thời ..."
9,https://www.topcv.vn/brand/fptis/tuyen-dung/ai-engineer-j2026003.html,AI Engineer,ai engineer,data_science_ml,Hà Nội,machine learning,ml_concepts,requirements,required,Kinh nghiệm AI/ML ứng dụng thực tế


,job_family,skill,importance,job_count
21,ai_automation_solutions,python,required,6
25,ai_automation_solutions,sql,required,5
12,ai_automation_solutions,llm,mentioned,4
1,ai_automation_solutions,api,required,3
8,ai_automation_solutions,excel,required,2
11,ai_automation_solutions,git,required,2
13,ai_automation_solutions,llm,required,2
14,ai_automation_solutions,machine learning,mentioned,2
15,ai_automation_solutions,machine learning,required,2
19,ai_automation_solutions,problem solving,required,2


Tỷ lệ job không extract được skill: 0.11653116531165311


In [97]:
def format_salary_brief(row):
    mn = row.get("salary_min_vnd_month")
    mx = row.get("salary_max_vnd_month")
    if pd.notna(mn) and pd.notna(mx):
        return f"{int(mn):,}-{int(mx):,} VND/tháng"
    if pd.notna(mn):
        return f"từ {int(mn):,} VND/tháng"
    if row.get("salary_is_negotiable") is True:
        return "thỏa thuận"
    return ""


def build_job_text_sparse(row):
    parts = []

    for field in [
        row.get("job_title_canonical"),
        row.get("job_family"),
        row.get("seniority_final"),
        row.get("tags_text_phobert"),
        row.get("skills_text_phobert"),
        row.get("skills_required_text_phobert"),
        row.get("requirements_clean_strict"),
        row.get("description_clean_strict"),
    ]:
        s = safe_str(field)
        if s and s != "unknown":
            parts.append(s)

    return "\n".join(parts).strip()


def build_job_text_phobert_match(row):
    parts = []

    title = row.get("job_title_for_phobert")
    family = row.get("job_family")
    seniority = row.get("seniority_final")
    required_skills = row.get("skills_required_text_phobert")
    preferred_skills = row.get("skills_preferred_text_phobert")
    exp = row.get("experience_min_years")
    location = row.get("location_norm")
    work_mode = row.get("work_mode")
    req = truncate_by_words(row.get("requirements_clean_phobert"), 160)

    if title:
        parts.append(f"Vị trí: {title}")
    if family and family != "unknown":
        parts.append(f"Nhóm nghề: {family}")
    if seniority and seniority != "unknown":
        parts.append(f"Cấp bậc: {seniority}")
    if required_skills:
        parts.append(f"Kỹ năng bắt buộc: {required_skills}")
    if preferred_skills:
        parts.append(f"Kỹ năng ưu tiên: {preferred_skills}")
    if exp is not None and not pd.isna(exp):
        parts.append(f"Kinh nghiệm tối thiểu: {exp} năm")
    if location:
        parts.append(f"Địa điểm: {location}")
    if work_mode and work_mode != "unknown":
        parts.append(f"Hình thức làm việc: {work_mode}")
    if req:
        parts.append(f"Yêu cầu chính: {req}")

    return "\n".join(parts).strip()


def build_job_text_phobert_chatbot(row):
    parts = []

    if row.get("job_title_for_phobert"):
        parts.append(f"Vị trí tuyển dụng: {row['job_title_for_phobert']}")
    if row.get("job_family") and row["job_family"] != "unknown":
        parts.append(f"Nhóm công việc: {row['job_family']}")
    if row.get("seniority_final") and row["seniority_final"] != "unknown":
        parts.append(f"Cấp bậc: {row['seniority_final']}")
    if row.get("location_norm"):
        parts.append(f"Địa điểm: {row['location_norm']}")
    if row.get("work_mode") and row["work_mode"] != "unknown":
        parts.append(f"Hình thức làm việc: {row['work_mode']}")
    salary_brief = format_salary_brief(row)
    if salary_brief:
        parts.append(f"Mức lương: {salary_brief}")
    if row.get("skills_required_text_phobert"):
        parts.append(f"Kỹ năng bắt buộc: {row['skills_required_text_phobert']}")
    if row.get("skills_preferred_text_phobert"):
        parts.append(f"Kỹ năng ưu tiên: {row['skills_preferred_text_phobert']}")
    if row.get("requirements_clean_phobert"):
        parts.append(f"Yêu cầu:\n{truncate_by_words(row['requirements_clean_phobert'], 220)}")
    if row.get("description_clean_phobert"):
        parts.append(f"Mô tả công việc:\n{truncate_by_words(row['description_clean_phobert'], 220)}")
    if row.get("benefits_clean_phobert"):
        parts.append(f"Quyền lợi:\n{truncate_by_words(row['benefits_clean_phobert'], 120)}")

    return "\n\n".join(parts).strip()

In [98]:
df_clean["job_text_sparse"] = df_clean.apply(build_job_text_sparse, axis=1)
df_clean["job_text_phobert_match"] = df_clean.apply(build_job_text_phobert_match, axis=1)
df_clean["job_text_phobert_chatbot"] = df_clean.apply(build_job_text_phobert_chatbot, axis=1)

df_clean["dense_encoder_route"] = "phobert"
df_clean["dense_similarity_metric"] = "cosine"

display(df_clean[
    [
        "job_title_raw",
        "job_family",
        "seniority_final",
        "job_text_sparse",
        "job_text_phobert_match",
        "job_text_phobert_chatbot"
    ]
].head(3))

,job_title_raw,job_family,seniority_final,job_text_sparse,job_text_phobert_match,job_text_phobert_chatbot
0,Junior FP&A Analyst - Hồ Chí Minh,data_analytics,unknown,"fp&a analyst\ndata_analytics\nchuyên môn data analyst, tài chính, kế toán\ncommunication\ncommunication\n- at least 2 year experience in fthe inance/accounting area within big/complex organization...",Vị trí: Junior FP A Analyst - Hồ Chí Minh\nNhóm nghề: data_analytics\nKỹ năng bắt buộc: communication\nKinh nghiệm tối thiểu: 2.0 năm\nĐịa điểm: Hồ Chí Minh\nYêu cầu chính: - At least 2 year exper...,"Vị trí tuyển dụng: Junior FP A Analyst - Hồ Chí Minh\n\nNhóm công việc: data_analytics\n\nĐịa điểm: Hồ Chí Minh\n\nMức lương: 12,000,000-20,000,000 VND/tháng\n\nKỹ năng bắt buộc: communication\n\n..."
1,Data Governance Specialist,data_governance_quality,unknown,"data governance specialist\ndata_governance_quality\nchuyên môn data analyst\nsql, etl, data warehouse\n- đã tốt nghiệp đh trở lên ưu tiên các chuyên ngành về công nghệ thông tin/ hệ thống thông t...","Vị trí: Data Governance Specialist\nNhóm nghề: data_governance_quality\nKỹ năng ưu tiên: sql, etl, data warehouse\nKinh nghiệm tối thiểu: 2.0 năm\nĐịa điểm: Hà Nội\nYêu cầu chính: - Đã tốt nghiệp ...","Vị trí tuyển dụng: Data Governance Specialist\n\nNhóm công việc: data_governance_quality\n\nĐịa điểm: Hà Nội\n\nMức lương: 20,000,000-30,000,000 VND/tháng\n\nKỹ năng ưu tiên: sql, etl, data wareho..."
2,Project Manager Dự Án AI HUB,product_project_ba,unknown,ai project manager\nproduct_project_ba\nchuyên môn it project manager\ncommunication\ncommunication\ntối thiểu 2-3 năm kinh nghiệm ở vị trí product manager project manager hoặc operations manager....,Vị trí: Project Manager Dự Án AI HUB\nNhóm nghề: product_project_ba\nKỹ năng bắt buộc: communication\nKinh nghiệm tối thiểu: 2.0 năm\nĐịa điểm: Hà Nội\nYêu cầu chính: Tối thiểu 2-3 năm kinh nghiệm...,"Vị trí tuyển dụng: Project Manager Dự Án AI HUB\n\nNhóm công việc: product_project_ba\n\nĐịa điểm: Hà Nội\n\nMức lương: 30,000,000-35,000,000 VND/tháng\n\nKỹ năng bắt buộc: communication\n\nYêu cầ..."


In [99]:
def split_long_text(text, max_chars=700, overlap=80):
    text = clean_text_preserve_structure(text)
    if not text:
        return []

    paras = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks = []
    current = ""

    for para in paras:
        if len(current) + len(para) + 2 <= max_chars:
            current = f"{current}\n\n{para}".strip()
        else:
            if current:
                chunks.append(current)
            if len(para) <= max_chars:
                current = para
            else:
                start = 0
                while start < len(para):
                    end = min(start + max_chars, len(para))
                    chunk = para[start:end]
                    chunks.append(chunk.strip())
                    start = max(end - overlap, end)
                current = ""

    if current:
        chunks.append(current)

    return [c.strip() for c in chunks if c.strip()]


SECTION_PRIORITY = {
    "title": 5.0,
    "requirements": 4.5,
    "description": 4.0,
    "benefits": 3.0,
    "company": 2.5,
}


def build_chunk_text_phobert(row, section_type: str, chunk_text: str):
    title = safe_str(row.get("job_title_for_phobert"))
    family = safe_str(row.get("job_family"))
    seniority = safe_str(row.get("seniority_final"))

    section_map = {
        "title": "Tiêu đề",
        "requirements": "Yêu cầu",
        "description": "Mô tả công việc",
        "benefits": "Quyền lợi",
        "company": "Giới thiệu công ty",
    }
    section_label = section_map.get(section_type, section_type)

    parts = []
    if title:
        parts.append(f"Vị trí: {title}")
    if family and family != "unknown":
        parts.append(f"Nhóm nghề: {family}")
    if seniority and seniority != "unknown":
        parts.append(f"Cấp bậc: {seniority}")
    parts.append(f"Phần: {section_label}")
    parts.append(truncate_by_words(chunk_text, 180))
    return "\n".join([p for p in parts if p]).strip()


def build_job_section_records(row):
    rows = []

    section_map = {
        "title": safe_str(row.get("job_title_for_phobert")),
        "requirements": safe_str(row.get("requirements_clean_phobert")),
        "description": safe_str(row.get("description_clean_phobert")),
        "benefits": safe_str(row.get("benefits_clean_phobert")),
        "company": safe_str(row.get("company_description_clean_phobert")),
    }

    for section_type, text in section_map.items():
        if not text:
            continue

        chunks = [text] if section_type == "title" else split_long_text(text, max_chars=700, overlap=80)
        for chunk_order, chunk_text in enumerate(chunks):
            rows.append({
                "job_url": row.get("job_url"),
                "job_title_display": row.get("job_title_display"),
                "job_title_canonical": row.get("job_title_canonical"),
                "job_family": row.get("job_family"),
                "job_family_source": row.get("job_family_source"),
                "seniority_final": row.get("seniority_final"),
                "location_norm": row.get("location_norm"),
                "work_mode": row.get("work_mode"),
                "section_type": section_type,
                "chunk_order": chunk_order,
                "section_priority": SECTION_PRIORITY.get(section_type, 1.0),
                "chunk_text_raw": chunk_text,
                "chunk_text_phobert": build_chunk_text_phobert(row, section_type, chunk_text),
            })

    return rows


section_rows = []
for _, row in df_clean.iterrows():
    section_rows.extend(build_job_section_records(row))

job_sections_df = pd.DataFrame(section_rows)
display(job_sections_df.head(10))
print("job_sections_df shape:", job_sections_df.shape)

,job_url,job_title_display,job_title_canonical,job_family,job_family_source,seniority_final,location_norm,work_mode,section_type,chunk_order,section_priority,chunk_text_raw,chunk_text_phobert
0,https://www.topcv.vn/brand/beeogisticscorporation/tuyen-dung/junior-fpa-analyst-ho-chi-minh-j1851113.html,Junior FP&A Analyst - Hồ Chí Minh,fp&a analyst,data_analytics,title+description,unknown,Hồ Chí Minh,unknown,title,0,5.0,Junior FP A Analyst - Hồ Chí Minh,Vị trí: Junior FP A Analyst - Hồ Chí Minh\nNhóm nghề: data_analytics\nPhần: Tiêu đề\nJunior FP A Analyst - Hồ Chí Minh
1,https://www.topcv.vn/brand/beeogisticscorporation/tuyen-dung/junior-fpa-analyst-ho-chi-minh-j1851113.html,Junior FP&A Analyst - Hồ Chí Minh,fp&a analyst,data_analytics,title+description,unknown,Hồ Chí Minh,unknown,requirements,0,4.5,"- At least 2 year experience in fthe inance/accounting area within big/complex organizations and/or audit services in reputable finance consulting firms. - Bachelor s degree in Finance/Accounting,...",Vị trí: Junior FP A Analyst - Hồ Chí Minh\nNhóm nghề: data_analytics\nPhần: Yêu cầu\n- At least 2 year experience in fthe inance/accounting area within big/complex organizations and/or audit servi...
2,https://www.topcv.vn/brand/beeogisticscorporation/tuyen-dung/junior-fpa-analyst-ho-chi-minh-j1851113.html,Junior FP&A Analyst - Hồ Chí Minh,fp&a analyst,data_analytics,title+description,unknown,Hồ Chí Minh,unknown,description,0,4.0,"- Overseeing all financial planning, reporting analysis for the Bee office. - Track management reports focusing on Budget/Estimate/Actuals to ensure those are delivered on time and with high-quali...","Vị trí: Junior FP A Analyst - Hồ Chí Minh\nNhóm nghề: data_analytics\nPhần: Mô tả công việc\n- Overseeing all financial planning, reporting analysis for the Bee office. - Track management reports ..."
3,https://www.topcv.vn/brand/beeogisticscorporation/tuyen-dung/junior-fpa-analyst-ho-chi-minh-j1851113.html,Junior FP&A Analyst - Hồ Chí Minh,fp&a analyst,data_analytics,title+description,unknown,Hồ Chí Minh,unknown,description,1,4.0,"res, and internal control processes. - Make recommendations for the improvement optimization of the efficiency of operation based on the results of financial control testing. - Prepare regular and...","Vị trí: Junior FP A Analyst - Hồ Chí Minh\nNhóm nghề: data_analytics\nPhần: Mô tả công việc\nres, and internal control processes. - Make recommendations for the improvement optimization of the eff..."
4,https://www.topcv.vn/brand/beeogisticscorporation/tuyen-dung/junior-fpa-analyst-ho-chi-minh-j1851113.html,Junior FP&A Analyst - Hồ Chí Minh,fp&a analyst,data_analytics,title+description,unknown,Hồ Chí Minh,unknown,benefits,0,3.0,"- Competitive salary according to personal experience and ability - Lunch allowance, phone allowance - Salary increase and annual bonus on holidays (2/9, 1/5, Mid-Autumn Festival, 1/6, New Year, L...","Vị trí: Junior FP A Analyst - Hồ Chí Minh\nNhóm nghề: data_analytics\nPhần: Quyền lợi\n- Competitive salary according to personal experience and ability - Lunch allowance, phone allowance - Salary..."
5,https://www.topcv.vn/brand/beeogisticscorporation/tuyen-dung/junior-fpa-analyst-ho-chi-minh-j1851113.html,Junior FP&A Analyst - Hồ Chí Minh,fp&a analyst,data_analytics,title+description,unknown,Hồ Chí Minh,unknown,company,0,2.5,"Xuất phát với ước mơ xây dựng một doanh nghiệp Việt, có thể cung cấp, phát triển dịch vụ logistics toàn cầu, tin cậy dựa trên chất lượng dịch vụ, con người và công nghệ, cùng niềm đam mê với nghề ...","Vị trí: Junior FP A Analyst - Hồ Chí Minh\nNhóm nghề: data_analytics\nPhần: Giới thiệu công ty\nXuất phát với ước mơ xây dựng một doanh nghiệp Việt, có thể cung cấp, phát triển dịch vụ logistics t..."
6,https://www.topcv.vn/brand/beeogisticscorporation/tuyen-dung/junior-fpa-analyst-ho-chi-minh-j1851113.html,Junior FP&A Analyst - Hồ Chí Minh,fp&a analyst,data_analytics,title+description,unknown,Hồ Chí Minh,unknown,company,1,2.5,"g), Cambodia, 

job_sections_df shape: (2931, 13)


In [100]:
def maybe_segment_vi_text(text):
    text = safe_str(text)
    if not text:
        return ""
    if HAS_UNDERTHESEA:
        try:
            return word_tokenize(text, format="text")
        except Exception:
            return text
    return text


def prepare_phobert_text(text: str) -> str:
    text = clean_text_for_phobert(text)
    text = maybe_segment_vi_text(text)
    return text if text else "[EMPTY]"


def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)


def cosine_similarity_matrix(query_emb, doc_embs):
    return np.dot(query_emb, doc_embs.T)

In [ ]:
import os
from pathlib import Path
import torch
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel

HF_CACHE_DIR = Path("./hf_cache")
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ========= CONFIG =========
# Khi đã tải model xong 1 lần, nên bật True để chỉ dùng local cache
# USE_LOCAL_ONLY = False

# Nếu muốn ép offline hoàn toàn sau khi đã cache sẵn model
# HF_OFFLINE = False

USE_LOCAL_ONLY = True
HF_OFFLINE = True
# Tùy chọn: giảm thời gian chờ khi gọi metadata từ HF Hub
os.environ["HF_HUB_ETAG_TIMEOUT"] = "10"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

if HF_OFFLINE:
    os.environ["HF_HUB_OFFLINE"] = "1"


def load_phobert_model(model_name, cache_dir, local_only=False):
    """
    Load tokenizer + model theo thứ tự ưu tiên:
    1) local-only nếu được yêu cầu
    2) online với cache_dir
    3) fallback local-only nếu online fail
    """
    print(f"[INFO] Loading tokenizer/model from: {model_name}")
    print(f"[INFO] Cache dir: {cache_dir.resolve()}")
    print(f"[INFO] local_files_only={local_only}")

    if local_only:
        tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            cache_dir=str(cache_dir),
            local_files_only=True
        )
        model = AutoModel.from_pretrained(
            model_name,
            cache_dir=str(cache_dir),
            local_files_only=True
        )
        return tokenizer, model

    try:
        tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            cache_dir=str(cache_dir)
        )
        model = AutoModel.from_pretrained(
            model_name,
            cache_dir=str(cache_dir)
        )
        return tokenizer, model

    except Exception as e:
        print("[WARN] Online load failed. Trying local cache only...")
        print("[WARN] Reason:", repr(e))

        tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            cache_dir=str(cache_dir),
            local_files_only=True
        )
        model = AutoModel.from_pretrained(
            model_name,
            cache_dir=str(cache_dir),
            local_files_only=True
        )
        return tokenizer, model


if RUN_EMBEDDING:
    try:
        tokenizer, model = load_phobert_model(
            model_name=PHOBERT_MODEL_NAME,
            cache_dir=HF_CACHE_DIR,
            local_only=USE_LOCAL_ONLY
        )

        model.to(DEVICE)
        model.eval()

        print(f"[INFO] Device: {DEVICE}")
        print("[INFO] PhoBERT loaded successfully.")

        def encode_phobert_texts(
            texts,
            batch_size=PHOBERT_BATCH_SIZE,
            max_length=PHOBERT_MAX_LENGTH_MATCH
        ):
            if texts is None or len(texts) == 0:
                hidden_size = getattr(model.config, "hidden_size", 768)
                return np.empty((0, hidden_size), dtype=np.float32)

            embeddings = []
            prepared = [prepare_phobert_text(t) for t in texts]

            for i in tqdm(range(0, len(prepared), batch_size), desc="Encoding"):
                batch = prepared[i:i + batch_size]

                encoded = tokenizer(
                    batch,
                    padding=True,
                    truncation=True,
                    max_length=max_length,
                    return_tensors="pt"
                )

                input_ids = encoded["input_ids"].to(DEVICE)
                attention_mask = encoded["attention_mask"].to(DEVICE)

                with torch.no_grad():
                    output = model(
                        input_ids=input_ids,
                        attention_mask=attention_mask
                    )

                batch_emb = mean_pooling(output, attention_mask)

                if NORMALIZE_EMBEDDINGS:
                    batch_emb = F.normalize(batch_emb, p=2, dim=1)

                embeddings.append(batch_emb.cpu().numpy().astype(np.float32))

            return np.vstack(embeddings)

        # test nhanh để chắc model encode được
        _test_emb = encode_phobert_texts(["data analyst"], batch_size=1, max_length=16)
        print(f"[INFO] Test embedding shape: {_test_emb.shape}")

    except Exception as e:
        print("[ERROR] Failed to load or initialize PhoBERT.")
        print("Reason:", repr(e))
        raise

else:
    print("[INFO] RUN_EMBEDDING=False -> skip model loading.")

[INFO] Loading tokenizer/model from: vinai/phobert-base
[INFO] Cache dir: D:\TTTN\project_v2\notebooks\preprocessing\hf_cache
[INFO] local_files_only=False


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/phobert-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
lm_head.dense.bias              | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
lm_head.decoder.weight          | UNEXPECTED |  | 
lm_head.decoder.bias            | UNEXPECTED |  | 
lm_head.layer_norm.weight       | UNEXPECTED |  | 
lm_head.layer_norm.bias         | UNEXPECTED |  | 
lm_head.dense.weight            | UNEXPECTED |  | 
lm_head.bias                    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[INFO] Device: cpu
[INFO] PhoBERT loaded successfully.


Encoding: 100%|██████████| 1/1 [00:00<00:00,  6.85it/s]

[INFO] Test embedding shape: (1, 768)


In [102]:
job_dense_embeddings = None

df_clean["embedding_row_id"] = np.arange(len(df_clean))

if RUN_EMBEDDING:
    job_dense_embeddings = encode_phobert_texts(
        df_clean["job_text_phobert_match"].fillna("").tolist(),
        batch_size=PHOBERT_BATCH_SIZE,
        max_length=PHOBERT_MAX_LENGTH_MATCH,
    )
    df_clean["has_dense_embedding"] = 1
    print("job_dense_embeddings shape:", job_dense_embeddings.shape)
else:
    df_clean["has_dense_embedding"] = 0
    print("[INFO] Skip job-level embedding.")

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Encoding: 100%|██████████| 24/24 [00:58<00:00,  2.44s/it]

job_dense_embeddings shape: (369, 768)


In [103]:
section_dense_embeddings = None

if RUN_EMBEDDING and RUN_SECTION_EMBEDDING and len(job_sections_df) > 0:
    job_sections_df["section_embedding_row_id"] = np.arange(len(job_sections_df))
    section_dense_embeddings = encode_phobert_texts(
        job_sections_df["chunk_text_phobert"].fillna("").tolist(),
        batch_size=PHOBERT_BATCH_SIZE,
        max_length=PHOBERT_MAX_LENGTH_CHUNK,
    )
    job_sections_df["has_dense_embedding"] = 1
    print("section_dense_embeddings shape:", section_dense_embeddings.shape)
else:
    job_sections_df["has_dense_embedding"] = 0
    print("[INFO] Skip section-level embedding.")

Encoding: 100%|██████████| 184/184 [07:16<00:00,  2.37s/it]

section_dense_embeddings shape: (2931, 768)


In [104]:
def encode_query_for_matching(query: str, max_length=PHOBERT_MAX_LENGTH_MATCH):
    q = encode_phobert_texts([query], batch_size=1, max_length=max_length)
    return q[0]


def retrieve_top_jobs(query: str, top_k: int = 10):
    if job_dense_embeddings is None:
        raise RuntimeError("job_dense_embeddings is None. Hãy bật RUN_EMBEDDING=True")

    q_emb = encode_query_for_matching(query, max_length=PHOBERT_MAX_LENGTH_MATCH)
    scores = job_dense_embeddings @ q_emb
    top_idx = np.argsort(scores)[::-1][:top_k]

    cols = [
        "job_url", "job_title_display", "job_family", "location_norm",
        "work_mode", "skills_required", "skills_preferred", "job_text_phobert_match"
    ]
    out = df_clean.iloc[top_idx][cols].copy()
    out["cosine_score"] = scores[top_idx]
    return out.reset_index(drop=True)


def retrieve_top_sections(query: str, top_k: int = 10):
    if section_dense_embeddings is None:
        raise RuntimeError("section_dense_embeddings is None. Hãy bật RUN_SECTION_EMBEDDING=True")

    q_emb = encode_query_for_matching(query, max_length=PHOBERT_MAX_LENGTH_CHUNK)
    scores = section_dense_embeddings @ q_emb
    top_idx = np.argsort(scores)[::-1][:top_k]

    cols = [
        "job_url", "job_title_display", "job_family", "section_type",
        "chunk_order", "section_priority", "chunk_text_raw"
    ]
    out = job_sections_df.iloc[top_idx][cols].copy()
    out["cosine_score"] = scores[top_idx]
    return out.reset_index(drop=True)


# Demo
demo_query = "Data Analyst cần SQL Power BI và kinh nghiệm phân tích dữ liệu"
try:
    display(retrieve_top_jobs(demo_query, top_k=5))
    display(retrieve_top_sections(demo_query, top_k=5))
except Exception as e:
    print("[INFO] Demo retrieval skipped:", e)

Encoding: 100%|██████████| 1/1 [00:00<00:00, 16.27it/s]


,job_url,job_title_display,job_family,location_norm,work_mode,skills_required,skills_preferred,job_text_phobert_match,cosine_score
0,https://www.topcv.vn/viec-lam/data-analyst-linh-vuc-edtech/2054071.html,Data Analyst (Lĩnh Vực EdTech),data_analytics,Hồ Chí Minh,unknown,"[power bi, tableau, sql, excel]",[],"Vị trí: Data Analyst (Lĩnh Vực EdTech)\nNhóm nghề: data_analytics\nKỹ năng bắt buộc: power bi, tableau, sql, excel\nKinh nghiệm tối thiểu: 3.0 năm\nĐịa điểm: Hồ Chí Minh\nYêu cầu chính: Tốt nghiệp...",0.755841
1,https://www.topcv.vn/viec-lam/data-analyst/1985499.html,Data Analyst,data_analytics,Hà Nội,unknown,"[sql, power bi, tableau]",[python],"Vị trí: Data Analyst\nNhóm nghề: data_analytics\nCấp bậc: manager\nKỹ năng bắt buộc: sql, power bi, tableau\nKỹ năng ưu tiên: python\nKinh nghiệm tối thiểu: 0.0 năm\nĐịa điểm: Hà Nội\nYêu cầu chín...",0.755434
2,https://www.topcv.vn/viec-lam/senior-data-science-analysis/2061195.html,Senior Data Science & Analysis,data_science_ml,Hà Nội,unknown,"[sql, gcp, bigquery, power bi, python, pandas, machine learning]",[],"Vị trí: Senior Data Science Analysis\nNhóm nghề: data_science_ml\nKỹ năng bắt buộc: sql, gcp, bigquery, power bi, python, pandas, machine learning\nKinh nghiệm tối thiểu: 5.0 năm\nĐịa điểm: Hà Nội...",0.754211
3,https://www.topcv.vn/viec-lam/junior-data-integration-engineer/2060720.html,Junior Data Integration Engineer,data_engineering,Hà Nội,unknown,"[etl, sql, oracle, data warehouse]",[etl],"Vị trí: Junior Data Integration Engineer\nNhóm nghề: data_engineering\nKỹ năng bắt buộc: etl, sql, oracle, data warehouse\nKỹ năng ưu tiên: etl\nKinh nghiệm tối thiểu: 1.0 năm\nĐịa điểm: Hà Nội\nY...",0.749212
4,https://www.topcv.vn/viec-lam/chuyen-vien-cao-cap-mo-hinh-hoa-va-phan-tich-nang-cao-senior-data-scientist-modeling-and-advanced-analytics-khoi-du-lieu-holt-08/2067007.html,Chuyên Viên Cao Cấp Mô Hình Hóa Và Phân Tích Nâng Cao - Senior Data Scientist - Modeling And Advanced Analytics - Khối Dữ Liệu (HOLT.08),data_science_ml,Hà Nội,unknown,"[statistics, python, sql, spark, sas, r, aws, gcp, azure]",[],Vị trí: Chuyên Viên Cao Cấp Mô Hình Hóa Và Phân Tích Nâng Cao - Senior Data Scientist - Modeling And Advanced Analytics - Khối Dữ Liệu (HOLT.08)\nNhóm nghề: data_science_ml\nKỹ năng bắt buộc: stat...,0.748768


Encoding: 100%|██████████| 1/1 [00:00<00:00, 19.69it/s]


,job_url,job_title_display,job_family,section_type,chunk_order,section_priority,chunk_text_raw,cosine_score
0,https://www.topcv.vn/brand/fptis/tuyen-dung/data-analyst-j2066152.html,Data Analyst,data_analytics,requirements,1,4.5,"odeling cơ bản\nCó kinh nghiệm làm việc với Database: Oracle, Sqlserver, PortgreSQL\nHiểu Machine Learning cơ bản\nKinh nghiệm xây dựng Data Pipeline\nTư duy phân tích và logic tốt\nKhả năng đọc h...",0.787787
1,https://www.topcv.vn/viec-lam/senior-data-science-analysis/2061195.html,Senior Data Science & Analysis,data_science_ml,requirements,0,4.5,"Ít nhất 5 năm trong lĩnh vực Data Analysis/Data Science, ưu tiên ứng viên có kinh nghiệm trong ngành Y tế, Bán lẻ hoặc các hệ thống có quy trình khách hàng phức tạp.\nKỹ năng kỹ thuật:\nNền tảng d...",0.785489
2,https://www.topcv.vn/viec-lam/chuyen-vien-phan-tich-du-lieu-da/1861349.html,Chuyên Viên Phân Tích Dữ Liệu (DA),data_analytics,requirements,0,4.5,"Bằng đại học chuyên ngành CNTT, Khoa học dữ liệu, Thống kê hoặc tương đương\nKiến thức về phân tích dữ liệu và thống kê\nHiểu biết về các công cụ BI (Power BI, Tableau, v.v.)\nKiến thức về SQL và ...",0.770951
3,https://www.topcv.vn/brand/sapovn/tuyen-dung/data-analyst-j2042140.html,Data Analyst,data_analytics,requirements,0,4.5,"Trình độ và kinh nghiệm Tốt nghiệp Đại học trở lên các chuyên ngành: Khoa học Dữ liệu, Toán - Thống kê, Công nghệ Thông tin, Hệ thống Thông tin hoặc các ngành liên quan. Có tối thiểu 1-2 năm kinh ...",0.770131
4,https://www.topcv.vn/viec-lam/data-analyst/1985499.html,Data Analyst,data_analytics,requirements,0,4.5,Sử dụng thành thạo ngôn ngữ truy vấn dữ liệu Microsoft SQL.\nCó kinh nghiệm làm việc với Python trong xử lý và phân tích dữ liệu (ưu tiên).\nCó kinh nghiệm sử dụng các công cụ phân tích và trực qu...,0.769383


In [105]:
DOWNSTREAM_FIELD_GUIDE = {
    "tfidf_input": "job_text_sparse",
    "phobert_matching_input": "job_text_phobert_match",
    "phobert_chatbot_input": "job_text_phobert_chatbot",
    "chatbot_chunk_table": "job_sections_df",
    "chatbot_chunk_text_field": "chunk_text_phobert",
    "skill_table": "job_skill_map_df",
    "role_skill_stats": "role_skill_stats_df",
    "job_family_field": "job_family (final: title + description)",
    "seniority_field": "seniority_final (from job_level_raw)",
    "retrieval_metric": "cosine_similarity_on_l2_normalized_embeddings",
}

pd.Series(DOWNSTREAM_FIELD_GUIDE)

tfidf_input                                               job_text_sparse
phobert_matching_input                             job_text_phobert_match
phobert_chatbot_input                            job_text_phobert_chatbot
chatbot_chunk_table                                       job_sections_df
chatbot_chunk_text_field                               chunk_text_phobert
skill_table                                              job_skill_map_df
role_skill_stats                                      role_skill_stats_df
job_family_field                  job_family (final: title + description)
seniority_field                      seniority_final (from job_level_raw)
retrieval_metric            cosine_similarity_on_l2_normalized_embeddings
dtype: str

In [106]:
matching_cols = [
    "job_url", "job_id",
    "job_title_display", "job_title_canonical", "job_family",
    "location_norm", "location_city", "location_district", "work_mode",
    "salary_min_vnd_month", "salary_max_vnd_month", "salary_is_negotiable",
    "experience_min_years", "experience_max_years", "experience_type",
    "education_level_norm", "employment_type_norm", "job_level_norm", "seniority_from_title",
    "skills_extracted", "skills_required", "skills_preferred", "skill_groups",
    "job_text_sparse", "job_text_phobert_match", "job_text_phobert_chatbot",
    "embedding_row_id", "has_dense_embedding", "dense_encoder_route", "dense_similarity_metric",
]

matching_cols = [c for c in matching_cols if c in df_clean.columns]
df_matching_ready = df_clean[matching_cols].copy()
df_matching_ready["dense_model_name"] = PHOBERT_MODEL_NAME
df_matching_ready["dense_similarity_metric"] = "cosine"

display(df_matching_ready.head(3))
print(df_matching_ready.shape)

,job_url,job_id,job_title_display,job_title_canonical,job_family,location_norm,location_city,location_district,work_mode,salary_min_vnd_month,salary_max_vnd_month,salary_is_negotiable,experience_min_years,experience_max_years,experience_type,education_level_norm,employment_type_norm,job_level_norm,skills_extracted,skills_required,skills_preferred,skill_groups,job_text_sparse,job_text_phobert_match,job_text_phobert_chatbot,embedding_row_id,has_dense_embedding,dense_encoder_route,dense_similarity_metric,dense_model_name
0,https://www.topcv.vn/brand/beeogisticscorporation/tuyen-dung/junior-fpa-analyst-ho-chi-minh-j1851113.html,None,Junior FP&A Analyst - Hồ Chí Minh,fp&a analyst,data_analytics,Hồ Chí Minh,Hồ Chí Minh,quận Tân,unknown,12000000.0,20000000.0,False,2.0,2.0,fixed,bachelor,full_time,unknown,[communication],[communication],[],[soft_skills],"fp&a analyst\ndata_analytics\nchuyên môn data analyst, tài chính, kế toán\ncommunication\ncommunication\n- at least 2 year experience in fthe inance/accounting area within big/complex organization...",Vị trí: Junior FP A Analyst - Hồ Chí Minh\nNhóm nghề: data_analytics\nKỹ năng bắt buộc: communication\nKinh nghiệm tối thiểu: 2.0 năm\nĐịa điểm: Hồ Chí Minh\nYêu cầu chính: - At least 2 year exper...,"Vị trí tuyển dụng: Junior FP A Analyst - Hồ Chí Minh\n\nNhóm công việc: data_analytics\n\nĐịa điểm: Hồ Chí Minh\n\nMức lương: 12,000,000-20,000,000 VND/tháng\n\nKỹ năng bắt buộc: communication\n\n...",0,1,phobert,cosine,vinai/phobert-base
1,https://www.topcv.vn/brand/educa/tuyen-dung/data-governance-specialist-j2055051.html,None,Data Governance Specialist,data governance specialist,data_governance_quality,Hà Nội,Hà Nội,quận Thanh,unknown,20000000.0,30000000.0,False,2.0,2.0,fixed,bachelor,full_time,unknown,"[sql, etl, data warehouse]",[],"[sql, etl, data warehouse]","[database, data_engineering, data_modeling]","data governance specialist\ndata_governance_quality\nchuyên môn data analyst\nsql, etl, data warehouse\n- đã tốt nghiệp đh trở lên ưu tiên các chuyên ngành về công nghệ thông tin/ hệ thống thông t...","Vị trí: Data Governance Specialist\nNhóm nghề: data_governance_quality\nKỹ năng ưu tiên: sql, etl, data warehouse\nKinh nghiệm tối thiểu: 2.0 năm\nĐịa điểm: Hà Nội\nYêu cầu chính: - Đã tốt nghiệp ...","Vị trí tuyển dụng: Data Governance Specialist\n\nNhóm công việc: data_governance_quality\n\nĐịa điểm: Hà Nội\n\nMức lương: 20,000,000-30,000,000 VND/tháng\n\nKỹ năng ưu tiên: sql, etl, data wareho...",1,1,phobert,cosine,vinai/phobert-base
2,https://www.topcv.vn/brand/educa/tuyen-dung/project-manager-du-an-ai-hub-j2067747.html,None,Project Manager Dự Án AI HUB,ai project manager,product_project_ba,Hà Nội,Hà Nội,quận Thanh,unknown,30000000.0,35000000.0,False,2.0,2.0,fixed,bachelor,full_time,unknown,[communication],[communication],[],[soft_skills],ai project manager\nproduct_project_ba\nchuyên môn it project manager\ncommunication\ncommunication\ntối thiểu 2-3 năm kinh nghiệm ở vị trí product manager project manager hoặc operations manager....,Vị trí: Project Manager Dự Án AI HUB\nNhóm nghề: product_project_ba\nKỹ năng bắt buộc: communication\nKinh nghiệm tối thiểu: 2.0 năm\nĐịa điểm: Hà Nội\nYêu cầu chính: Tối thiểu 2-3 năm kinh nghiệm...,"Vị trí tuyển dụng: Project Manager Dự Án AI HUB\n\nNhóm công việc: product_project_ba\n\nĐịa điểm: Hà Nội\n\nMức lương: 30,000,000-35,000,000 VND/tháng\n\nKỹ năng bắt buộc: communication\n\nYêu cầ...",2,1,phobert,cosine,vinai/phobert-base


(369, 30)


In [107]:
chatbot_cols = [
    "job_url", "job_id",
    "job_title_display", "job_title_canonical",
    "job_family", "job_family_source",
    "seniority_final",
    "location_norm", "work_mode",
    "salary_min_vnd_month", "salary_max_vnd_month",
    "experience_min_years", "experience_max_years",
    "skills_extracted", "skills_required", "skills_preferred",
    "requirements_clean_phobert", "description_clean_phobert", "benefits_clean_phobert",
    "job_text_phobert_chatbot",
    "embedding_row_id", "has_dense_embedding"
]

chatbot_cols = [c for c in chatbot_cols if c in df_clean.columns]
df_chatbot_ready = df_clean[chatbot_cols].copy()

display(df_chatbot_ready.head(3))
print(df_chatbot_ready.shape)

,job_url,job_id,job_title_display,job_title_canonical,job_family,job_family_source,seniority_final,location_norm,work_mode,salary_min_vnd_month,salary_max_vnd_month,experience_min_years,experience_max_years,skills_extracted,skills_required,skills_preferred,requirements_clean_phobert,description_clean_phobert,benefits_clean_phobert,job_text_phobert_chatbot,embedding_row_id,has_dense_embedding
0,https://www.topcv.vn/brand/beeogisticscorporation/tuyen-dung/junior-fpa-analyst-ho-chi-minh-j1851113.html,None,Junior FP&A Analyst - Hồ Chí Minh,fp&a analyst,data_analytics,title+description,unknown,Hồ Chí Minh,unknown,12000000.0,20000000.0,2.0,2.0,[communication],[communication],[],"- At least 2 year experience in fthe inance/accounting area within big/complex organizations and/or audit services in reputable finance consulting firms. - Bachelor s degree in Finance/Accounting,...","- Overseeing all financial planning, reporting analysis for the Bee office. - Track management reports focusing on Budget/Estimate/Actuals to ensure those are delivered on time and with high-quali...","- Competitive salary according to personal experience and ability - Lunch allowance, phone allowance - Salary increase and annual bonus on holidays (2/9, 1/5, Mid-Autumn Festival, 1/6, New Year, L...","Vị trí tuyển dụng: Junior FP A Analyst - Hồ Chí Minh\n\nNhóm công việc: data_analytics\n\nĐịa điểm: Hồ Chí Minh\n\nMức lương: 12,000,000-20,000,000 VND/tháng\n\nKỹ năng bắt buộc: communication\n\n...",0,1
1,https://www.topcv.vn/brand/educa/tuyen-dung/data-governance-specialist-j2055051.html,None,Data Governance Specialist,data governance specialist,data_governance_quality,title+description,unknown,Hà Nội,unknown,20000000.0,30000000.0,2.0,2.0,"[sql, etl, data warehouse]",[],"[sql, etl, data warehouse]","- Đã tốt nghiệp ĐH trở lên, ưu tiên các chuyên ngành về Công nghệ thông tin/ Hệ thống thông tin quản lý hoặc các lĩnh vực liên quan và tối thiểu 2 năm kinh nghiệm trong lĩnh vực Data Governance/ D...","- Xây dựng, triển khai và duy trì khung Data Governance cho toàn hệ thống dữ liệu của Công ty. - Xây dựng chính sách, tiêu chuẩn, quy định về quản lý dữ liệu (data quality, data profiling, data se...","Mức lương thỏa thuận theo năng lực. Thử việc hưởng 100% lương trong 2 tháng. Thưởng Lễ/Tết, thưởng hiệu quả làm việc: theo quy định của Công ty. Chính sách đánh giá điều chỉnh lương hàng năm minh ...","Vị trí tuyển dụng: Data Governance Specialist\n\nNhóm công việc: data_governance_quality\n\nĐịa điểm: Hà Nội\n\nMức lương: 20,000,000-30,000,000 VND/tháng\n\nKỹ năng ưu tiên: sql, etl, data wareho...",1,1
2,https://www.topcv.vn/brand/educa/tuyen-dung/project-manager-du-an-ai-hub-j2067747.html,None,Project Manager Dự Án AI HUB,ai project manager,product_project_ba,title_priority,unknown,Hà Nội,unknown,30000000.0,35000000.0,2.0,2.0,[communication],[communication],[],"Tối thiểu 2-3 năm kinh nghiệm ở vị trí Product Manager, Project Manager hoặc Operations Manager. Ưu tiên ứng viên từng tham gia các dự án automation, AI, data analytics hoặc chuyển đổi số. Ưu tiên...",1. Lập kế hoạch Xây dựng chiến lược AI (20%) Xây dựng roadmap triển khai AI Hub nhằm nâng cao năng suất các bộ phận trong công ty. Phân tích quy trình vận hành hiện tại của các phòng ban để xác đị...,"Tinh thần: - Làm việc trong môi trường start-up có tốc độ tăng trưởng và phát triển nhanh. - Được chia sẻ cơ hội học tập chất lượng cao đến trẻ em toàn Việt Nam - Môi trường làm việc: Trẻ trung, n...","Vị trí tuyển dụng: Project Manager Dự Án AI HUB\n\nNhóm công việc: product_project_ba\n\nĐịa điểm: Hà Nội\n\nMức lương: 30,000,000-35,000,000 VND/tháng\n\nKỹ năng bắt buộc: communication\n\nYêu cầ...",2,1


(369, 22)


In [108]:
section_cols = [
    "section_embedding_row_id",
    "job_url", "job_title_display", "job_title_canonical",
    "job_family", "location_norm", "work_mode",
    "section_type", "chunk_order", "section_priority",
    "chunk_text_raw", "chunk_text_phobert",
    "has_dense_embedding"
]
section_cols = [c for c in section_cols if c in job_sections_df.columns]

job_sections_ready = job_sections_df[section_cols].copy()
display(job_sections_ready.head(3))
print(job_sections_ready.shape)

,section_embedding_row_id,job_url,job_title_display,job_title_canonical,job_family,location_norm,work_mode,section_type,chunk_order,section_priority,chunk_text_raw,chunk_text_phobert,has_dense_embedding
0,0,https://www.topcv.vn/brand/beeogisticscorporation/tuyen-dung/junior-fpa-analyst-ho-chi-minh-j1851113.html,Junior FP&A Analyst - Hồ Chí Minh,fp&a analyst,data_analytics,Hồ Chí Minh,unknown,title,0,5.0,Junior FP A Analyst - Hồ Chí Minh,Vị trí: Junior FP A Analyst - Hồ Chí Minh\nNhóm nghề: data_analytics\nPhần: Tiêu đề\nJunior FP A Analyst - Hồ Chí Minh,1
1,1,https://www.topcv.vn/brand/beeogisticscorporation/tuyen-dung/junior-fpa-analyst-ho-chi-minh-j1851113.html,Junior FP&A Analyst - Hồ Chí Minh,fp&a analyst,data_analytics,Hồ Chí Minh,unknown,requirements,0,4.5,"- At least 2 year experience in fthe inance/accounting area within big/complex organizations and/or audit services in reputable finance consulting firms. - Bachelor s degree in Finance/Accounting,...",Vị trí: Junior FP A Analyst - Hồ Chí Minh\nNhóm nghề: data_analytics\nPhần: Yêu cầu\n- At least 2 year experience in fthe inance/accounting area within big/complex organizations and/or audit servi...,1
2,2,https://www.topcv.vn/brand/beeogisticscorporation/tuyen-dung/junior-fpa-analyst-ho-chi-minh-j1851113.html,Junior FP&A Analyst - Hồ Chí Minh,fp&a analyst,data_analytics,Hồ Chí Minh,unknown,description,0,4.0,"- Overseeing all financial planning, reporting analysis for the Bee office. - Track management reports focusing on Budget/Estimate/Actuals to ensure those are delivered on time and with high-quali...","Vị trí: Junior FP A Analyst - Hồ Chí Minh\nNhóm nghề: data_analytics\nPhần: Mô tả công việc\n- Overseeing all financial planning, reporting analysis for the Bee office. - Track management reports ...",1


(2931, 13)


In [109]:
artifact_paths = {}

artifact_paths["jobs_matching_ready_v3"] = save_table(
    df_matching_ready, ARTIFACT_DIR / "jobs_matching_ready_v3"
)

artifact_paths["jobs_chatbot_ready_v3"] = save_table(
    df_chatbot_ready, ARTIFACT_DIR / "jobs_chatbot_ready_v3"
)

artifact_paths["jobs_chatbot_sections_v3"] = save_table(
    job_sections_ready, ARTIFACT_DIR / "jobs_chatbot_sections_v3"
)

artifact_paths["job_skill_map_v3"] = save_table(
    job_skill_map_df, ARTIFACT_DIR / "job_skill_map_v3"
)

artifact_paths["role_skill_stats_v3"] = save_table(
    role_skill_stats_df, ARTIFACT_DIR / "role_skill_stats_v3"
)

artifact_paths["job_embedding_index_v3"] = save_table(
    df_clean[["embedding_row_id", "job_url", "job_title_display", "job_text_phobert_match"]],
    ARTIFACT_DIR / "job_embedding_index_v3"
)

if len(job_sections_df) > 0:
    artifact_paths["job_section_embedding_index_v3"] = save_table(
        job_sections_df[
            ["section_embedding_row_id", "job_url", "job_title_display", "section_type", "chunk_order", "chunk_text_raw"]
        ],
        ARTIFACT_DIR / "job_section_embedding_index_v3"
    )

if RUN_EMBEDDING and job_dense_embeddings is not None:
    job_emb_path = ARTIFACT_DIR / "job_dense_embeddings_v3.npy"
    np.save(job_emb_path, job_dense_embeddings)
    artifact_paths["job_dense_embeddings_v3"] = str(job_emb_path)

if RUN_EMBEDDING and RUN_SECTION_EMBEDDING and section_dense_embeddings is not None:
    section_emb_path = ARTIFACT_DIR / "section_dense_embeddings_v3.npy"
    np.save(section_emb_path, section_dense_embeddings)
    artifact_paths["section_dense_embeddings_v3"] = str(section_emb_path)

print("[INFO] Saved main artifacts.")
pd.Series(artifact_paths)

[INFO] Saved main artifacts.


jobs_matching_ready_v3                    ..\outputs_preprocessing_v3\jobs_matching_ready_v3.parquet
jobs_chatbot_ready_v3                      ..\outputs_preprocessing_v3\jobs_chatbot_ready_v3.parquet
jobs_chatbot_sections_v3                ..\outputs_preprocessing_v3\jobs_chatbot_sections_v3.parquet
job_skill_map_v3                                ..\outputs_preprocessing_v3\job_skill_map_v3.parquet
role_skill_stats_v3                          ..\outputs_preprocessing_v3\role_skill_stats_v3.parquet
job_embedding_index_v3                    ..\outputs_preprocessing_v3\job_embedding_index_v3.parquet
job_section_embedding_index_v3    ..\outputs_preprocessing_v3\job_section_embedding_index_v3.parquet
job_dense_embeddings_v3                      ..\outputs_preprocessing_v3\job_dense_embeddings_v3.npy
section_dense_embeddings_v3              ..\outputs_preprocessing_v3\section_dense_embeddings_v3.npy
dtype: str

In [110]:
manifest = {
    "notebook_version": NOTEBOOK_VERSION,
    "run_timestamp": datetime.now().isoformat(),
    "raw_input_path": str(RAW_INPUT_PATH),
    "artifact_dir": str(ARTIFACT_DIR.resolve()),
    "n_jobs": int(len(df_clean)),
    "n_sections": int(len(job_sections_df)),
    "embedding_config": {
        "model_name": PHOBERT_MODEL_NAME,
        "metric": "cosine",
        "normalize_embeddings": NORMALIZE_EMBEDDINGS,
        "job_text_field": "job_text_phobert_match",
        "chatbot_text_field": "job_text_phobert_chatbot",
        "section_text_field": "chunk_text_phobert",
        "max_length_match": PHOBERT_MAX_LENGTH_MATCH,
        "max_length_chatbot": PHOBERT_MAX_LENGTH_CHATBOT,
        "max_length_chunk": PHOBERT_MAX_LENGTH_CHUNK,
        "batch_size": PHOBERT_BATCH_SIZE,
        "segmentation": "underthesea_if_available",
    },
    "downstream_field_guide": DOWNSTREAM_FIELD_GUIDE,

    # thêm block này
    "derived_fields": {
        "job_title_canonical": "normalized from job_title_raw after removing trailing noise, bracket noise, and synonym mapping",
        "job_family_from_title": "inferred from job_title_canonical using JOB_FAMILY_RULES",
        "job_family_from_description": "inferred from description_clean_strict using JOB_FAMILY_DESCRIPTION_HINTS",
        "job_family": "final resolved job family using title priority over description",
        "job_family_source": "source of final job_family: title / description / title+description / title_priority / unknown",
        "job_level_norm": "normalized from job_level_raw using JOB_LEVEL_RULES",
        "seniority_final": "final seniority derived from job_level_raw only",
        "seniority_source": "source of final seniority, expected to be job_level_raw or unknown",
    },

    # thêm block này nếu muốn rõ schema output
    "output_schema_notes": {
        "matching_table": "df_matching_ready uses job_family and seniority_final as final structured fields",
        "chatbot_table": "df_chatbot_ready uses job_family and seniority_final for job summary display",
        "section_table": "job_sections_ready uses job_family and seniority_final for chunk-level retrieval context",
    },

    "artifacts": artifact_paths,
}

manifest_path = ARTIFACT_DIR / "manifest_v3_phobert.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("[INFO] Manifest saved:", manifest_path)
display(pd.Series(manifest))

[INFO] Manifest saved: ..\outputs_preprocessing_v3\manifest_v3_phobert.json


notebook_version                                                                                                                                                                                         preprocessing_v3_phobert
run_timestamp                                                                                                                                                                                          2026-03-25T16:06:57.689278
raw_input_path                                                                                                                                                ..\..\data\raw\jobs\topcv_all_fields_merged_2026-03-21_19-27-47.csv
artifact_dir                                                                                                                                                                D:\TTTN\project_v2\notebooks\outputs_preprocessing_v3
n_jobs                                                                                          

In [111]:
length_stats = pd.DataFrame({
    "job_text_sparse_len": df_clean["job_text_sparse"].fillna("").str.len(),
    "job_text_phobert_match_len": df_clean["job_text_phobert_match"].fillna("").str.len(),
    "job_text_phobert_chatbot_len": df_clean["job_text_phobert_chatbot"].fillna("").str.len(),
})

display(length_stats.describe())

if len(job_sections_df) > 0:
    section_stats = (
        job_sections_df.assign(chunk_len=job_sections_df["chunk_text_phobert"].fillna("").str.len())
        .groupby("section_type")["chunk_len"]
        .describe()
        .reset_index()
    )
    display(section_stats)

print("Avg skills per job:", df_clean["skills_extracted"].apply(len).mean())
print("Jobs without extracted skills:", (df_clean["skills_extracted"].apply(len) == 0).sum())

,job_text_sparse_len,job_text_phobert_match_len,job_text_phobert_chatbot_len
count,369.000000,369.000000,369.000000
mean,1986.596206,863.842818,2296.753388
std,1204.604988,249.071280,631.088140
min,463.000000,275.000000,884.000000
25%,1287.000000,699.000000,1861.000000
50%,1778.000000,895.000000,2331.000000
75%,2345.000000,1021.000000,2712.000000
max,10669.000000,1455.000000,4270.000000


,section_type,count,mean,std,min,25%,50%,75%,max
0,benefits,548.0,568.025547,215.894532,82.0,406.00,583.5,777.00,924.0
1,company,682.0,597.652493,244.910552,83.0,385.25,744.0,798.75,933.0
2,description,714.0,617.201681,226.122833,85.0,448.00,754.5,797.00,930.0
3,requirements,618.0,597.469256,224.711073,86.0,417.50,697.0,783.00,922.0
4,title,369.0,126.192412,45.988873,65.0,96.00,117.0,140.00,389.0


Avg skills per job: 5.051490514905149
Jobs without extracted skills: 43


## Ghi chú sử dụng artifact

- `jobs_matching_ready_v3`: dùng cho matching/recommendation
  - trường chính: `job_text_phobert_match`
  - field cấu trúc chính:
    - `job_family` = final family
    - `seniority_final` = final seniority từ `job_level_raw`
  - vector tương ứng: `job_dense_embeddings_v3.npy`

- `jobs_chatbot_ready_v3`: dùng cho chatbot summary-level
  - trường chính: `job_text_phobert_chatbot`

- `jobs_chatbot_sections_v3`: dùng cho retrieval theo chunk/section
  - trường chính: `chunk_text_phobert`
  - vector tương ứng: `section_dense_embeddings_v3.npy`

- `job_skill_map_v3`: bảng long format job-skill
- `role_skill_stats_v3`: thống kê skill theo nhóm nghề
- `manifest_v3_phobert.json`: metadata toàn bộ pipeline